In [ ]:
import base64, csv, gzip, hashlib, importlib.util, json, os, platform, re, shutil, signal, subprocess, sys
import threading, time, types, urllib.error, urllib.request, zipfile, zlib
from datetime import datetime, timezone
from pathlib import Path

if not sys.platform.startswith("linux") or not Path("/kaggle/working").is_dir():
    raise RuntimeError("本 Notebook 应在 Kaggle 的 Linux 会话里运行。")

NOTEBOOK_START = time.time()

# ---------- 可以改的设置 ----------
TOTAL_BUDGET_HOURS = 11.2   # 到这个时间点停止所有实验，开始汇总打包（Kaggle 一次最长 12 小时）
MIN_START_MINUTES = 3       # 离停止时间不到这么多分钟时，不再开始新的一组
PARALLEL = 4                # 开思考的组：每张卡同时处理几条（OLLAMA_NUM_PARALLEL）
REQUEST_TIMEOUT = 1800      # 单个请求最多等几秒
N_SUBSET = 200              # 和上次一样，这些组只跑验证集前 200 条
OLLAMA_VERSION = "0.40.3"
OLLAMA_MODELS = ["qwen3.5:0.8b", "qwen3.5:2b", "qwen3.5:4b", "qwen3.5:9b"]
EXPECTED_DIGESTS = {  # 上次运行时 Ollama 报告的模型版本
    "qwen3.5:0.8b": "de63045f29755d4721c09d54c9b45c2dadb644f4fd7d604d4cac90d96049971c",
    "qwen3.5:2b": "0689d44085e06d165161a8a9a1731344278cfb5aade63a3c3dbdb48ab54b130a",
    "qwen3.5:4b": "d8b0f5e9760cd1682034f292d7ef72ec46f432149be0df7574bf2d6e92e38c04",
    "qwen3.5:9b": "56671c2ab9385f9cfcb404638e32cd62d88e3501d44822208363c010179a3c90",
}
# 上次运行的原始返回，已提交到 GitHub；按固定提交下载，校验 SHA-256 后作为断点
CHECKPOINT_COMMIT = "1d01fce4a162adcc3a67ac7fbab6d9bc4bd94ae2"
CHECKPOINT_URLS = [
    "https://raw.githubusercontent.com/Pan-Yan-0/study_note/{commit}/ML_note/SST2/metrics/raw/{name}",
    "https://cdn.jsdelivr.net/gh/Pan-Yan-0/study_note@{commit}/ML_note/SST2/metrics/raw/{name}",
]
CHECKPOINTS = {  # 文件名 -> SHA-256、字节数、可用条数
    "ollama_qwen3.5-0.8b_basic_think-off_all_8808a639c2ba.jsonl.gz": {"sha256": "c15a3d1abd19d2b788b37c6656fe9a88e00c25d496b724b30383ed168c715c82", "bytes": 51397, "records": 872},
    "ollama_qwen3.5-0.8b_basic_think-on_n200_b64dc64a59d6.jsonl.gz": {"sha256": "dc8416f64c162399c71782ecf604829e6b421232cf92ba57eed7e8067ae83d85", "bytes": 296154, "records": 152},
    "ollama_qwen3.5-0.8b_cot_think-on_n200_0e246b02592a.jsonl.gz": {"sha256": "002da57851b848615150db75211fd17f2371b5a6169bce299c25f0ffc0ac7fee", "bytes": 327518, "records": 109},
    "ollama_qwen3.5-2b_basic_think-off_all_7aaee91ff0a4.jsonl.gz": {"sha256": "af40f7a3e744166746dabd048483d89b095f1a3ed8023f8ab9f5b39c36d52606", "bytes": 51406, "records": 872},
    "ollama_qwen3.5-2b_basic_think-on_n200_d9de6d47a35c.jsonl.gz": {"sha256": "3e05b80059534356cc49dcd8593753652f0a7214433947cc4f4cce6013b0079e", "bytes": 231814, "records": 200},
    "ollama_qwen3.5-2b_cot_think-on_n200_3b785a2f31a2.jsonl.gz": {"sha256": "38290e32a269e8e6ce8c50b2636de49ca84ad796f0eb4be287d3a7733fb7366d", "bytes": 235722, "records": 86},
    "ollama_qwen3.5-4b_basic_think-off_all_052bf7f44efd.jsonl.gz": {"sha256": "ecd4d788edc65aefc148445814ef1a74429bf7f4a706715c099a77fd834a507f", "bytes": 51387, "records": 872},
    "ollama_qwen3.5-4b_basic_think-on_n200_f99f49af8375.jsonl.gz": {"sha256": "b9c965342d6f364c1e770a448151639629282605b12ecfdaefe1f18429f082f2", "bytes": 144527, "records": 200},
    "ollama_qwen3.5-4b_cot_think-off_n200_3b6d1cb8adef.jsonl.gz": {"sha256": "67ad66961f7fa696fae21a298f2e2d961c19b735c7a6e7f5b9e2f84925103c0f", "bytes": 45546, "records": 200},
    "ollama_qwen3.5-4b_cot_think-on_n200_feef13cbaf24.jsonl.gz": {"sha256": "e562d2e06466eaff3484cd7e0933986beb0be652cc316b75acae04ef8728d7de", "bytes": 151554, "records": 86},
    "ollama_qwen3.5-4b_role_cot_think-off_n200_818616711ace.jsonl.gz": {"sha256": "55ebb5b3d97aabb92b735fd6881045162d15f583130631e7c732d39b3afd0c18", "bytes": 44789, "records": 200},
    "ollama_qwen3.5-4b_role_cot_think-on_n200_5be799124791.jsonl.gz": {"sha256": "ad89f31ef455f4144dd3cd2f21258ee9ac71fc4156dd1711d044b1c610280ce8", "bytes": 153279, "records": 99},
    "ollama_qwen3.5-9b_basic_think-off_all_592c71504568.jsonl.gz": {"sha256": "80ce8763f6651defed395872ff9b0cb754061ead68f50f7851709f6e350c12a2", "bytes": 51454, "records": 872},
    "ollama_qwen3.5-9b_basic_think-on_n200_5194c28620c6.jsonl.gz": {"sha256": "9c2be102295d7df8a1e77e229dec1d24cca87b6430c415eebcf92597fe1b4621", "bytes": 35445, "records": 60},
    "ollama_qwen3.5-9b_cot_think-on_n200_7bb4e18c58f6.jsonl.gz": {"sha256": "7145d61553248523422220842dea951625665d9e8ddc505807aee75ecf45b6d6", "bytes": 65423, "records": 22},
}
KAGGLE_INPUT = Path("/kaggle/input")  # 把上一次运行的 Output 加为 Input 后，里面的断点也会被用上

# ---------- 路径 ----------
ROOT = Path("/kaggle/working/sst2_finish")
DATA_DIR, META_DIR, SCRIPT_DIR = ROOT / "data", ROOT / "metadata", ROOT / "scripts"
OLLAMA_RESULTS, RESCORED = ROOT / "ollama_results", ROOT / "rescored"
MODEL_DIR = Path("/tmp/sst2_ollama_models")  # 模型权重放在这里，不进结果包
for directory in (ROOT, DATA_DIR, META_DIR, SCRIPT_DIR, OLLAMA_RESULTS, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
EVAL_SCRIPT = SCRIPT_DIR / "sst2_instruction_eval.py"
RESCORE_SCRIPT = SCRIPT_DIR / "rescore_offline.py"
DATA = DATA_DIR / "sst2_validation.tsv"
ZIP_PATH = Path("/kaggle/working/sst2_finish_results.zip")
# GPU 0 的服务用上次的地址：接着跑的组的断点文件名里含有这个地址，必须在这个服务上跑
SERVER_PORTS = {"gpu0": 11434, "gpu1": 11435}
CONTEXT_PER_REQUEST = 8192  # 和上次一样，每个请求可用的上下文长度


# ---------- 这次要跑的 16 组 ----------
def experiment(model, prompt, think):
    return {"model": model, "prompt": prompt, "think": think, "n": N_SUBSET}


PLAN = ([experiment(m, p, "off") for m in ("qwen3.5:0.8b", "qwen3.5:2b", "qwen3.5:9b")
         for p in ("cot", "role_cot")]
        + [experiment(m, p, "on") for m, p in (
            ("qwen3.5:0.8b", "basic"), ("qwen3.5:0.8b", "cot"), ("qwen3.5:0.8b", "role_cot"),
            ("qwen3.5:2b", "cot"), ("qwen3.5:2b", "role_cot"), ("qwen3.5:4b", "cot"),
            ("qwen3.5:4b", "role_cot"), ("qwen3.5:9b", "basic"), ("qwen3.5:9b", "cot"),
            ("qwen3.5:9b", "role_cot"))])
PLAN = [e for e in PLAN if e["model"] in OLLAMA_MODELS]

# 上次单路运行时每条平均用的秒数，只用来估计各组的工作量、决定先跑哪组
SECONDS_PER_ROW = {
    ("qwen3.5:0.8b", "on"): {"basic": 23.5, "cot": 32.9, "role_cot": 32.9},
    ("qwen3.5:2b", "on"): {"basic": 17.9, "cot": 41.8, "role_cot": 41.8},
    ("qwen3.5:4b", "on"): {"basic": 15.4, "cot": 41.7, "role_cot": 36.2},
    ("qwen3.5:9b", "on"): {"basic": 30.4, "cot": 156.5, "role_cot": 156.5},
    ("qwen3.5:0.8b", "off"): {"cot": 3.0, "role_cot": 3.0},
    ("qwen3.5:2b", "off"): {"cot": 2.5, "role_cot": 2.5},
    ("qwen3.5:4b", "off"): {"cot": 3.0, "role_cot": 3.2},
    ("qwen3.5:9b", "off"): {"cot": 5.0, "role_cot": 5.0},
}
DEADLINE = NOTEBOOK_START + TOTAL_BUDGET_HOURS * 3600


def hours_used():
    return (time.time() - NOTEBOOK_START) / 3600


def seconds_left():
    return DEADLINE - time.time()


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def tail(path, chars=3000):
    path = Path(path)
    return path.read_text(encoding="utf-8", errors="replace")[-chars:] if path.exists() else ""


STATUS = {"started_at_utc": datetime.now(timezone.utc).isoformat(), "notes": []}
STATUS_LOCK = threading.Lock()


def note(message):
    """打印一条重要信息，同时记进 metadata/notebook_status.json。"""
    print(message, flush=True)
    with STATUS_LOCK:
        STATUS["notes"].append({"hours": round(hours_used(), 3), "message": message})
        save_json(META_DIR / "notebook_status.json", STATUS)


print(f"这次要跑 {len(PLAN)} 组，最多用到第 {TOTAL_BUDGET_HOURS} 小时：")
for i, e in enumerate(PLAN, 1):
    print(f"  {i:>2}. {e['model']:<13} {e['prompt']:<9} 思考 {e['think']:<3}  前 {e['n']} 条")

In [ ]:
EMBEDDED_ASSETS = {
    "sst2_instruction_eval.py": {"sha256": "301d0bb32aafe27a525fe228b687aedc353411135fb71f8077298fc92089bed2", "bytes": 42199,
        "zlib_base64": "eNrVff13E0eW6O/+K2qanWMJJNkGMpPVGSWHCZ4NbwhhgZnZeUarla0W1iJLGrWEYWydY0IMtsGYJHxjwkf4GhJ/EBIMGPD/MuOW5J/yL7z7UdVd3WoJh91977zsLFZ3V926devWvbdu3bq15Rc9VavcM5gr9JiFY6J0ojJcLOzoMgzj4MFD0e2ifu7M+uo9e+HB+ovbzaXP6z+e/enV9fXns83FhcbqqfqjO/bX8OacPfd44+bt9ecT9cUfuUpz8W/2vYdcwJ66V7+8YL9+ghBOTdY/vwVVGqu3mot37DOnG3cWG+fP/GPis64ue+mF/fKivfTKfj5Vv3mhfvnaPyZm+qLrr9caFx854JprN5p3zumY/WMCkajPT68/n7bvfW6vTTYfnGwun6pf+gHQ7RKiLyYAwsbjc82lkxs3Totj6Xwuk67kioWfXk29/2vo5807DKF++0xj4c1Pr6YBoX/Z+4d+0bj+uaiYVkVAPf5Yv7q0cf1C88ocfILq9uQjeCOifVAJYEBr8B9jv3HmnLBK+VwlYSCIgfj23qQhgFjNU68b82ebb76yz7xEWixeAZrCbw85hNgeE9yk6O3pE/XZu/WpCw5kRCza1wPNTvWKhGj+cGvj5l0R/YAwOdcHr+oL38hXfdwhALkjJpo/LjdPztj3/lY/f9+e+6Z+5Vl9/lt7+Q2iMHNb/KYynCsc/QCK66XWX11vnnlszzyy7wOuM3Ik4M3KE0AEBq65dBNe1m+uuTSYuNmcOLX+/Lx94QFQX0SjBFnYrybsyaf18w/tqWc/vbrh6ZBsXABFGzd+gKYbqwv26Wv2+Vv2w7P1y2fq898Bvtji6Wv1p5eIa2DQGw9n7QvnG98u8XBnTLNkmeZRIXbDr4P4y168Wr/8QuzavwfwAwqIT0tmYdcewb0DGm7MTwDTiFKuJHIFq5LO50URiqRzQDoAWczn0yNp6hagMP9SfEovGJi9eApROn+rcfEWAxQ96VKuZ2g4XaHZsbS+el92m4iCbDP3GIgCwwtjbr/8SrZCLWIr9uSz9dXLCkt78pW9+KI+DwMEfHqdmo4NlUrEnfQUtczyMbP8j4mTez8RByvVTK4Iv4/thafGwjRMrefPcQ598RomDbT+p1whUxy1xP7iqFk+OGzm88S9SLx/AkEQ393fv/9gf//vU0Cw1O/7/wzcZFhHo7FYzBAB/21BpsbZ+/ore3q28fgK8hVMpktTAJCFirCsyvYUkrZcHcKZlzJhFsZKJ4AvMuZIsRXgFw5nbHz2jT13f/3NWfgXp8v8WXvxHEzu5tpF+8bXm2qiILb3tjQxPYtvYe5vAkSbbk8+2jj1SEgRsilMeBJAGQepLaI+/RWwB/MGTq1Xc5sCVSoXj+UyZllxZzQ6UsyYefGXUbOwI/ZefOegbOS/BVpv7P1B52H7oNbKpsCXzVKxXPGSrz4zI8qmVc1XLMESoLH6Vf3r+fqTMyDqQJ/Yr2437zzq6qovzTVWP7cXvwYZbs8/ciqBSFhffWbfe7H+/CxOPqwEWuAxv2eG/o19YdZeuPJB7D+tYiHvtA4gYcjqt1dgQuPs1TgKp+eFb3k4mmtnGo/Osmy3L5xDaXl1CdRZ47MXAB7f3F7B6jSzNs7MNle+AFmJkuvmBPzW2x+yjmm95/ahj6DMAFJj/hZ0j8U9zNyNu6hs4Qe0BvITRDkoBnhsPvym/jVMixfAI/g4cQUwgh+V4lGzIOqXluE39KCxcFFrOWVVR0bS5RNEAZAtrpY5Wb9yG2YpayKuXV9Zqa+cbtz628bVH13AE6vwFe2Crmy5OCJSqWy1Ui2bqZTIjdC4pguFYoVUqtXVpd6Vj5TSZctUz0AA9XM4bQ3nc4PqERFTv4uW+lV2alrFoaNmxXk64RSpDJfNdCZXOOK8yI04tarlPLQRM8vlYtn3zoOXfFc2/1IFNc09HIJZYBIPW6qLHxWrhYpZVt8LQ9Vy2SxUYkwKp9jv9hw4eCj10aef7N/bf6h/d0QcIhz3F4v5/uPmULVSLEfEaDonGyqlK0gJVXs/PHZ1bRGJ4P/QlGHTR2nbqebZJ8DuoH3g38bkw/rM5yw4gWfbQen6uH/X7v4DINVDxm/NfHFU5CwYQaHNXKBsugKa1Boq5wahc2lRSVtHI4BurmxmxGiuMsw1StUKl5XywxJGl8vlRrZargyDUAF6VczjlZj4UzlXMQEcUKwExDW5croE1UvlXLpi5k9A4ZFS3gSbCT6aQo5LzAh3Hfp4z77fpw4e6t8vcc8WywCsYIEaAyaICJauQ+myma3mAVJ6sEj4mYJgYMfShYwYghEhLKyKWYoOnojiXw/ihDwo8FxBFLMAoFg9MgwCp1IUZsGqYqMiXzySG0rnCWB6CLgBQapuEba7Dv6e8NxVSOdP/NUkPI7kjsGEQloIGn8gIoilXAGrHsuZowQuA50vj+BLrGIBlwFbF4DGxXwa6HcipuFqHDCB/wrYGSBB3hSF6sggivGsMHNE+2gfAe2L8LDB46AJRUXBPAIz9pjJX+VLbFADXipaOSrjIIE969p/APj70EHo3ViXlOXEdI3vvhPbxcadH0F/N1aviY9gcP9l/yE0VBrTUyBqUWiCAfzgJEg+FNSXlkFCgWzFRcUDFG8bZ2YIojGYtnJDRlyEJLNuE8bhwuHCli1bxB6XT+OHCy3D9j9K7nckuiKvQ0/8GA0mumzCGR6X9IcL+4pCTSnzeCmfLpDMFcWy+M8q8HcWOJJewITGeYOTNYZka4GOdPxXOSeAiGPYiFkYMmviTzBm3VYbUvBsMImiH7YFfEDOgrgRjrTwx3vEH28eN84vSaWvOAM4oP7kM3tyCpeSC/fxf6SEmSGGihU/OxjwR5MJ7VkEy+Fs3ObBVisNYsxDBfWlY092Yk88rH0a11BouMzdbz78sjn9fXPxjb14nTtQLubN1H+lF10tJqjx52IV9CyyE/KDifozi8IWVD5J3REavTTOCCtnASMHU+KdyFHr6trd/8mnqYP9+w717/uon2QdSGBLZHP5EdGNLJhnm2AY1lUgO9NDlSpITIBpSSYlLkuPQAcsMYqaYGi4nLMqI2krWimbrbOCdEoezAGcNjmagKVyGlRGmsE5bVgFUGtx+F4qwuSIRkUuKzIgowdhsuLTSI7NFaHk2R/3wIC4Es1Qq0gYrTEURmYK7AR4MIYrlZIV78ElXkwVigGljYgwyGTGQupDNJsHU8eIBHRjtFg+apYtKP0+1MR5BroKnvre761xeYNN8UAMAAEgQzo/XLQq8b6+nTt2etp3rfPWtrWW+zwt/6rXbZmWom9v+f3e93t7jvXpbe8rFsyf22atk82zPSbWV8+zWQsSw567DNYqCs/61GUQrO2MnNSn+/b+ObXvD5/AmJZNHKFSLm+Gysa/D2yLJj/sC30YPxzr3Rb+8J+AA1K//fTf+nf7Sx4+PFg8bmYOjx22toaoVtitBu8O16DqFlhiX6zfmQLjvbm4DFKA7RHREyQ+e1zDBxbm7B2wnz9Al1TVBHvbGgKLBn0oWwQ6DeZmxN8n5jWZ+/eJm/ACzB5BE42VGLwUzbVrjSer62uLG/cBh6l/hfEX9pfndBfI1xvXJtnUx0X1qdfwo3F+GVY40j2U2rXv4J/6D6Q+2XXg92wdlg3obTYH0uOwtQ1+cs/G8+lBMz9eAt2SI9kUHpcfVN/GA6atIGAOScYVScJIUGub8/ghP8O/RJNxokg4/GEbmC4WEi1gK5Be47xEDLdFhJsoZseBlmF+ABqPI1XlozRoM/A0TmYE/Ah/2BYgjs44j8e4kpvjZB8D8sQmMHA5C8eKfsTlLwtsy3wGDAH5rN4n8C+MUmPxSv3qeVixrb+ex3Vdai9oCmRUA5nywzj8L2eNMxTAcNAcH3V+Qk+2xj8cj48n8JcBdXf9tn9v6k+fHiBWN0LKHBlXxsY4ew3HGw+X6l+fHGe34nj92RQ8ATHVdDJCvjmE3/r/bf/ePR/tOdQ657qYSocHUf342AxeUI/gL9bbpoZn6+EQldZQxm+HwzTzBmK/AJZN4s9/AuEDbe2BSYylaNr7J/LP6qcLz+mRotjP7pIPe0/PVHc7dQnMjflvm3fQxU4eNTSXwcportxiA9o+/aU9t4Jy5MFnyDV9xD4oGOlB9MGjEgewZFxfXcMPHlVOFa48Qxfv2rX67RX7NK4daahhGbv3030t8lMOflgXhXHuWssnKV739f/LrkN7/thPpDhIloIaCtQdTH/8JYeAeo69fQCfoIfoE7m01ri10Lj9Q33mKr+xJx+BldX47gE8NlaRIughv7a4MXGd3OMkAPu6Urt3Hfy4HxsFayo2kj5qVsogsEJjxuHq9u1921GxRbFteOzt2+F93Kk9ZrO9Gc9jXx8+9hk1MCG6MmaW3f0pWoWGUCDEsckw+uArVVjSDsBTBF8l46xmDQN9YNLtHYvFftMjPeBsGbP5C72oz080VqdYBdpTp+1XEzCowBBYhtzCAEb6D1dW7PlHwN31y2cAiPJooYBH9w22SquhBP8BRSLfgn1kqOYNNK0If0fa0QdaY0v1xgBiZepyyK0aEX1hp1aZl0mqcqxswpplyITiTmnDCMeAIrlSKKxgq2cXLz9aZIdzf7+/gzsEi+fs5SfNu9/Wpx7XLy+svzyN+znf3wESOe4w/m+QfAYRByXVD9WNzfTCQZeBedCVZQ2AwHDlN+YP8julSFEF8kcObIVxtqC8bMLjKELsGIwI3REYhrHl9+vPX1Jl0bzzqHHvpWMy1adpe2dqWW4Znaufm8YvF681F+/QLgo2Yx4H21l2WbIaSR19qU4Fs7myVUnRKlrAOoidmK1lyWV6HuZ1/eqStrInk0o2Iw2ssWhfjSyYh7O8s0MFwXKupOTCGhaF9fOPGhdOA0RWhLRhh6KRGwORx2oS12OaPUaQwPhULQp79lLjLLp8eS+yuTSLkLTartnWI1ldQiJQQEcwRtdXV+ufzzFG6H1buLL+fKFxbl7tH95AasxdAIpvTExv3HmBwurCA1gJQnPQln3h28bTGRik5ps3MBS6F1YOiBx2+utMOL90CQ/0JWMky2CZZYaknHOmTaFYkXXjflZmDjPMkVLlhKGNSkLkYfUVYms4BpZfJlcxyyGGEmbII1CKSg9E+5LYDlc182DTIlzV/Aj5OFwkBkZiZiETCseTclIY4nDlcPlwgfTe1v84bHQb4RZMQ2DnI7DYkXKxWgr1hcHeRwnM7fXBJDSofe4F8qQF+A3kC66s2bpVihnnzX945A7isXXLB9S+QNM6X6ClpZRFSHSCGwon/Y1oxfktoJovJN1ZQhSFDwO9SRCVsrkYCuMwlaVK1BVXDDsLF9BVlaHhEAHSKLNFbDy4DIzP8wAmhD03xdxbv/gCNKA990BOQ8mN5F0DS3L1YWN1ATQFV7dffwUQWLPwjFGM585xMsGhDzwI9A6Ili5XLPRqhWAcwnIg3IpIERQNiiYDffFk3GM1E9SISEniSHmIP/00CnvqAQoSoQS6gwrZPFj+BuqwEL+HFRHyG4lAZD5++4uE3ptwvMWC906K9Mhg7ki1WLUMPy9qUCJgwJC8NGgYzQKhbxF39skBdcWkwf2QNUfMynAx09p5CyfUW7nEmd4BpGg3z31dcipr9Gqp6kG2qxVmrkBhFIbUbCkrd6QQYmntajXQZ3G9LvMRl2rHSBIeymymdQh/vg0mlonlcT87FEbW81ucHuA+wgdrYXgXrItR/pPKY2eET+389Op6CLVsREjtjEEOZIlItXR1SYQkDQ36yEY9GHlnvwNDli0/0BONWzcaN95QcXeEqZJmy3nEBe55sMjAXrXKUh4j+gZwNPVqOILdtfpjGiRHpLsgNZm8Pdwy8bglt0AYJ6H3ZV/YNxOV5eTpa3AXNBhtO6KvRmOWmS6/e2c0Vvx/2iVneesf6YAuKeABuPdSE/ghiFzYyH+RXoGDH4DIO9IrGJKnS15RhTKqvXsRY6KWcH+7PrvY1plIUqOYzqTKxVErlC4fsTR5gTbTAHqhksFW+8CYkcscx84ox5BBYwzyxyBZEa5FBKz+krhgZUTqN+/XX15oLv0A0sU73bHtWCZdSaeysBx36YcbyTCQuIEc8pZxlWjGzKPjm4xJ4zDrLqwXs2Axmzuuy8+QEatYxxDPWOV4RUlnI6INR3oUjaCk84I2utBxHEKYEQEdLeLefMKoVrLR96PAFgCvYI4iWyVAjqHTP+vngHSG0BuyjsV2A0kP0ItQNuIin3B+tZgHyIdjAVSuxXKWVR20zEqIWwDb1sxnCukRUKxgPwwkA8wC64QVM4/DqjBrjGGPasJem9y4s2qfmwQDXqhWyMtAzcDi4J49dcXwooUGUS4iykhVs4D7MmisMxoBrQIkcr3GjgCyEv0wr9PDnpVmQNdhLepKhrJBvpjDmW1AB4BDQEC1heg3Fs+ReR0RvbjajXcFxTi10gA3vsYkEUmPQpdg8XaHAviYDEwmDJS59AOwM4cvTl1Wi0+YbbgUHYPCvyjXjNbeOJRNCEkGZ0QDu+6UB0lEUgg6iv1T7xXZ/lu66DTGRjUsuRt/exmAF0yPWLpUwiVPCIZfVYs4Q6AZtyTMcEmBNCGGSUXwf/gEgwSgarrs7cWXso7cR3ZeePvY2rf6ykrj4qP1l6dpVJB1cVSEs1RoLklXmgy7nXwGK1t4hKV5/fYrsGe0rv7VLBdTSO+EByVYoNCIU/GfXk1RIC3HcAqQKT+9msaNXgpnlSxxA6Mqb963l59wLC3UBlJTROojV95gxGJCCdM4zimXL8CEc9wyAXttPI3iajVJcwyRlotIfOd0ht5BiVoLHDmTLX1gXPFnFatl4lnAgyQgEwrBeTQnCefhbGokh4FJ3vEqwnAVjuXKxQKo4AoonXQ1D+z/8e9S/ft27/90z75DKNPUDuRwNspAaPvRHZdK+YQXLIU+oD4AoE6gEqkz+dIpbR4fMksVsYeK9Lci6DCU0Xj10l7+woVKUazX9QBa9QldlVOXmxOTHAaMr6OolzCSyV68DmNvzy9LzujUiwwOv451yCicqEZH8vmeI8B4pF+tynb4y1HXRGj6Gfb3r5/+UDgQLKzIndh4/UVjdX7jyuLG3auwKLafP28+PNluMq0/P9t8/VpwlLx970nzh/vQ+THcKLdvvLZPTzI0Gfo7cxu67QwWxmz9uCw8gyc2Ls3bp+Y4ztAIFFPCYCLKpp3odMKhJ2MeQ30tMA46kMq0mGw/l0AmlQfoIRn2zit47TwlNzPHeIpBNX6R1Gda8Jwi1ZixfNV+gW6HJA5NFMPZOeie5VSXX+9Bd9oLPpcPahTGT4sxJfCmvGH+QTH+01IsEl0JjnaCQCOqIwCyBgFl7tCb9xpxBfGB6I37xwT/DMS5QFI3ZvFDRDbS0aDdGRN6UHxbm3YIVhmW+B0wSp5mesiZFWHHgt2YnG28Xmw8WUVf8dIS0OuoeULAwz8mTq6/vrZx9yaqwWdPMXKUDgWg0F64as8/wje8NfH9HfvVxN8nHsD/mJgYJ7t0SY7D53PrL/AlB/zbJ+eludvVdfDQrkN/OJj6GKSeG5exs7cvLoxd+/cIRARbW3qB3tvv72AA8KUpGe6ws3c7FNMxRG/0+VVRyqcrwHMjntANYU9O2hOvnLo7oC6j11xcA5FQv3lq49oF5/NO+Awiy745gS3zIZcLs4wLOpO/majfug+8gpET0Wo5z2qW46kVkO2IH3AVBjzNfQZWEmJ591s6P3BFB0K1gJjwK308SvG4FsYOGBg1kRpNlwsmOkfMCm4uJVH9gGAM85cURsrgDocKk43thRfObgQVKcLEDo1YR1wniOsOIoveheRdAFpHlBGpsPDMQPkyls5kELzXQCqVUeJkjYH63IXGvZdJMQZFaiC5s/mqNZw4VK6aiCRzKJ+C+CifM5XnB/jDjWv48lzH0xJoxPDpD3m8AhSVPfcFk17xGi+OsiKVyhVylVQqZJl5WHKomPiIUEEwEZEu5VLAehFBAyM3kuApfTzFg4MbPyMltPKrtNHEoS6a8dmi2FTkM+H37qrY7Z9PEctgHk1SQe9isnjC3+4W6gsInXLOBN19fePkGdzHuHah/uPJ0M7t/xx2ZrZ97RGLhdB7x4/T68sLoPjqV57hZHv5lTw1tPIE2BUtylfX7c/XcPd66ZLTGoUSpdCiwSnuMXP1WO0YPNCPkBqJcAwDkHAFx8tVJyYJ7YC+7b+O9cL/UWBSPN5n+Gx/t9UBsqZSQ8RdqKwkOWK72fj6GL4eZ94LVcpVq5ICAy3xuzSoMx89GYQLgNkxJPkl4fCN6kDC5SnJIwn5NxJsAPj+00YpsTMitm51O+XDzGVjepSMS785YDqhcbrO1l4wOotzZZfPAUSbGeCFAaZEJqUa5b+g5w1p6eozkT6mBouZEzQZSTKhj0X6ZON+G8AHPtCx5GxPka1dIGHpEgHwKGpBlrSw1KhHBbSoQJgojhDaWL3aXLznHOJBg+HBSXvyqVoJo4fYPn0NOmBhBGImCLkxQ20wU9hd5UQJzS8DeAlrkK8GUGY3jIJj1GpdLWDw3FkKRwB3BVNHR9GWIJAMKqU1UyzUatgR/RCZ6BF8cAxsIg4jo6Nr2tBw+LyUkQw/TmPTSchJ9LSpEkNEYzLYH6NEYxLw1q0MtMVs1+RWDEyAgxXgMItEI9vxngaHgJFhhI+YUApWZOgItKhCCr+AYEBuaPEgUS0QKLrxEbBjlM7BMGimU9b4+NCh/WIMq9dg+Mf0+gP4NlkDZcOr4PW1O/WTeFhxDJeKZnggvqO3N1nD3UdcqJldLU3pGgomeV7XUCMlLYwAR8FFl8mIgtWJztQlgDFiWlb6iIncAUsBjE9GjqsCH6DcpDMbBQzS5FZqySDBZLjz3gGvSQJDEwXqu/aqpg1xpZxWM1Kf/WF9RlIp73hwJwcM+kRVSIwzPOBtVy3iyayZGXvxFJ6Wub1iv3iGh4nJEFh//VUn1rVKDmaSRzfBob9NZw7w2ZUA7S2lVojRRC9Oq7BRersN/wVQIVYqlkI6JcKb7QhF1GgBRXTK+15z6bSMQSIZgCHu51fr56dZiWPA1Xd3vEZJi5AnVekp45qdRnPtGu5TqxbZ+tVbFEHCDK2Lle8ZEw5mYSOcpTBvuzXXLjTvnNP6PzRczLFLEYgQ4yfclncKoEGbkMVicmZoRpEj691Qq4tv+DBi2ioWQJ6m5IQhfxZSUrdE7blv3ZKazJZvNEEFeMC0aYGqBBY6N9sU1Yto+yZV7InWABIggtMcXreIQVeLOLMfWonJJ4bsaTPudqJVPBhZsKetYSgk6crPKa4SUJ4ljStPFM6EbMT/XSIfAMfVKu1htZaR8Gru0oMUo2/p4TkCLtoe/QZxjy4D1urQPFiPZk8WZ4PQD4O/ZQniGonvvN6gOYnL0IQDzYky6AGls00YDtIu16ApDTWCjXAFUjPCvTy3RciVM9EKw0tBnvy43Hi6it7l1W8aF07jMdtXE42bE/zYfL20vvoMSrLnDffA177GaKvnM+yU4zPyEjkQmHmzTD4s7/HN2P5y8fiJj/l7aKwWphAl6sxmlglsX2mbWo5Ap10pX1uD1Vw+k+Kvoa0KKZ/13WJuq988UGg5e0ZWvvdjwGdRUZV7Napn8I1CdSQlQ+Jx5jpMUntXM7wEpFKsCApF2nmtpgaQpJU+Uv057BIhn3ACj/vGMtWRkhVCmOEY7RSCOqB9QiO8ufWPGKYdGisxZnzE0il6SNrM6VIpL0/C9WBjGKDr2aXURpX+gEz8i7sM04eHNinLgaYsdQP90VaoHEMHSygcy5ienoTf0XBDryjIKozP432GwhEz9J7PEEDa/c/adgJNZujZCJQjPR5Bo4Q4UTUnH2tdLXtyHZZjCvsBKRyTnVdigUYZ0RW4ybFniFOJn1rbgj4c9byV5pp+RDyGFny7xQSPYiWdQylqBo82qUWKMDSkSR+0WYmR9BmK29rZ20uGnyQCOcGpCWcf3jELOxGyg9lFcphNLOWcrV9cqp872cGoC4TvmmxZY8xls5poA9G1z6R0YR8niPwx7mLNu9fjXbcVKrlC1Xwb8XYGkyJgaSY1NQZCs8+ZiCH0jiDKk1NsNAJBZA6MUjWf9xQzNieZsoa/p0ErO2+fPkiI9yRDqJn/G7EjuI8om2JWHizS0Haxdauq8DPpyZQ6UKVoZ+8ylnGqxYXqRHAPgmbRHw7s7TSJoMM5ixyTyE00k9ASFCDCC5xw4YCZBSmVISBsypb5Dc0QWjHLWmE1UTbNCY5JIUcVMIfxuS4ZBHcq5h8pJrjFaVmUnWe/vAjmXPvtuaDtOoDmBONyWhwGrnKskMOlI3P832SFoKENceoNpQ0j4hD/4NGJvxPKm8TCu3HehmEN3kuy712Hlej66mzj82e8HasJF17Yoa7gYBKpGTmqZqz2tqUP1VGvwoErIKeY41pT5TqsiFyEMrDokEuiIOOnZVHk1pRfMM0OrBOriOAmF0MuDL1yTe4MwRKxmD9mosOagsncrRDXYnHjY+NqW9PrvZCHl7vaOWHVPqj0jLcU1D8Gt6AOKZPvAqMIlCdhojn1Haqm23c2Hp/DnUJYk+NG0NS39uIzuS155Vlz8Y3Yu/cTleXK374WjEGE0orK8S1AV1gc4GZkoqWGP4+W4Tmb4emzu5uDOmr6DUZIOS4HdMXKLU9YLrpZu1Cl6WorMEdAd+ekXuuvbwN4gGwcLnQHbOuDJhsayWACPdxZFH4oCR0Eej5OPd24tFafv7W+dtNeuCqrHcdDCC/O2hfOo1/81UT98jLLQ9cMkETHsWYexIN6cl/Gx4ObXAtL8TSUxenvnPAfUFDYOlMrYihC/OY8w/hCzQH3+DuXd1dtVFw9qtLqYHvyZ00J3c0QeucVv05Gfc80FLSN2X6Gh30Nd3XagOqMUKcohfdiApNlPZ9wMnt1DL5NVyvFlNtiSEo+dD2QSJLYBhwTyGW1FW4xm20l/87ef/6Ve6KQU36tv7jVWLzCZ7IEtYkxCpeuNZ89bfzt5caVH3Rq/2qnHGqFEo02Z7CRkS+923diE0Mw7zl1E+6Gagc18dwbHydbuL/x+J6cA9RrjbYhXyezsPz0nobojfW6PU7IHjMOvTGtl5xnbv35WdEr0El59SvcmwAJOv1V8/V3MOldf6deHpBee924dN9ePIX+GKktqgXqdcijJCQbyd+dxisiChSCpNSK7BDFF2PgooRaS42xFZ4a06DVeMkTHaM/8K0bltndJGPFb5zIo2x3YaxQ6655wsPx6GZ1EANl/31X9H+no3/tjf5zLBVNYsAsHfIFFFSsBGVXoVOBKdDgR8wyhzC8a4872ZBqkiqYDtMTkTxzLc4c0BEcBUcJN0a943FkUImN14vrz8/jydmXc5wfz558QpnuzjSXTvNUxSCek2v25Gz9ykM+elpfWWmu3QC5zhFDGBhzeZmjqdafL2gh7EjMTO4IuoX4PJZn0N2B15xE8igSZ95Kpa2hXC4h3RFWsVxBGWZRzAh61kppIE2xbCVCRoTcevopMjnwMv1czBpOb3/vVyEM1fW5oMKxYfO4RFM5cTCeHmuiMclfBgZU0FzEGycX0QLZkm6MGw5FkrHBIw65I+S+sYaGTcrysp0d3MRQ7JmRYtvx8CjnjsZbjguHnvyc4NiO5nG0Z2XergGtRjKiuaiDBL6eACbIfxzxbQFqasEPqZPnUh5XcKaWtFGJ4jV91kraMwHDEbeUOtyFEaJA0qGjpSJN0XRlOE7nIhxH2wDNI5oNzHZogcf9n9yAFHVMAgw0q2Lp603mVfqIPqHU4IkKHhR1w1vkWVFkMe0k6VEQrmYhY8lIJyckIZtFGwuklscTmCtkzOMR51Sle4iAzxwGbizSacq2ce+yoW0J5+hi64IxcJkW6IYrm0NqwrI7lBpvcYUGLS+p0v86+Om+3VSaFnUR8YdCDp+0d+EO7gSkDip37RRmVPR18gn8EQWK8glwNk9OGSr0EwAMeJvoo9j/+uwt++Ych4KCbQumAOdcxASkL55xuF77pfwWgWmTYYn6fAHaw12Ouceg8tdfnrZnZwA6xgZMTjUerm5cuwfN2MtvNk6hswF9cmuv7ZnbIE2bi8v260soeq9+X1+4u/78Hh5J+B6DLXHF8+OKfn5XPxYUG8Wki5I3gRMH4jz+ydZB1x199sr362uL9r1ZpkqMtK3gbNH1+cecdgH3aRRezaX7mNP01XVYlCO+tLMefDSCp4SDyCY8tpKvNecRsF1EbkfIAx/ej7wGQ/EcplMX7fbR34kdcHV5dYmzHOOmOSlKpoTRElTiIEMuMiOsDqrQypX9FQjj5p3GjUVOIoscEuyORVE1AACl4knSjvZQ12ZnN2KTHnWO5rPqy/Ap3EFMRuejEuA3+3T9NazovgX8BE5VTM/N+XIpPPpc49XL+uxdpMmLWxjyt/g3MCuB1yXj0rzB1DKY7vY8fMKwgZt3/MxK+zLEZ7QhY6QHAw+IkUxkhpb4enQD+hZkKpbh4mgKKFMsZ5AZtI0r12WhwlAPF/ApGo0KHkcBpE2MIZG74Vd3skZpp90MvYK/kYLvTqojTAoaZZmYu28vXMAQHyqpLAO3MHPFgOZG0g7J0/k8YcT+ExRYyF9MpirRBKresApAg4ZlWNFYN8ded6uT6iBMPxDwiS3j7m4/SsykGjqejp15uXHxmtMxKuv2yuuzcxJXOH3RM1hQW8qvlqRcFh17xilJsGWZT6JN92SyiuA++oFyHhWnO7h1SsM9JZ9xt5teYKDJWDcnYu5WpOoG7oK/le6kbIfDYbtbI5mJIykNc+6vJJw8Jjkds08H7q1ikCINGlRRliO1ifte1RE8tSFfGLqh6ZSWx/Td0thFKKqOygVUQcuqSgkaHQOIkjrwmSc6MagfF4QlFG7Dl72gWs+AwE9XzjttDGS5TAKPvtVoJ3CMcExEMbmSF+kEZaJw2oGWN7tXzCADIb4jQJX/IP42wiJQjwE7tnUrDjfu1kNtXxSMwamEh05gYE2xWsiE1Hj3QFGxM8wrWyfbiq+24oW4YpOIjqn81aZF0PspVdLfdKgANpWsHnbR+MDhr3YIkYCwXCpJ4eI5NO7lQR+ASrlaGAJj16E0a1LpRud0MDBBjlSGjc6AOPcSObrJAQ5TLCRzetPw4eeWORRuoa/kW6Kw/O1vyOeud/hDf51Em6W3M8ZBfntnzvs/bQagFlQ2nHZGhcSLR8XI48adgaWPHUlZoF/BenAYRmKnXvupyTy8w8/DateBzdQh65i2ZosIv7D06fChYp7CgNQRe5Y4EZ7xHOrAUyIiWQB/SO6hk26Mamts2dvGItJKT0y1QtoHfzGn64fpkl1Bx+SNUWPTZ+XRHnMKgNbCE1t5wUeneLsStzdfPGncdo8ujsrT9ERdsHC1UBj5rjgaQiqGu1oO5RHtPYR4i05/q0b3tet1oEhtgSkbpP7wCVViGed1RJ1bdDRgB+ntTnFqSnJAUu3n6VwbEcHTNWg/TwPfZk4i9LZTTKW8G4i/B8ZMR/BKcuoIK8ZKOgnm0LZRFz6EZFh+BMc/5fODeDbu6ADFAOijFquPjUrOX0OXcxh+bxrPQjQwBjTxmGw1idW9E2LMGuhWSqc7GY/tzJLFZWn2VK0HHwuO6aWnSKP6Uu+wba7vemXVqTpaVlFZ1kFc1MkRSF8c5RJsxbPNT1cqXWc3J9ofY8r0BxOlO8lGI/2syYukZNE2JbWCXry5jy2V3J5irfZYinZYbhrJd8FxcyjyfMAU7zCCkzwqngkGbBCh4XFHuGU6yTLu5SzEGje8CNqzlwAFyQIvnto3Jbu5ygo5bnsWFvUPvwzAFM/qrJ7GBYGaM9ryyNIC1tTWCsVoDbSogqT3cK82mQbqTx/VP59LiubiG8zD4NwJlM0iAShlzMaZc+urs+hd4LCq05P2Il44owdzoYtdhpYvNU+9tiefUgS9rNIe6UIHnPGo9s9DGrfF7NkfUEJcfGZPLXdAmPH0RYx58XRsvTarTwcN/+RFvkOH1NSy5n9Gn9vGtQvOlJfptmdu2/ceCs95WvvFMz6lny2b1rBgH4wHNWnHbgYvR9wIlxNlioBzuMlGdwVREt8v+ASFPfeYnWbNpUtIsJc3PE0r+x1Z6udh4sxLMN79iLnJQZ14A5mf5aODf6SIA6nq7akrnMcAL94il6cR1jb+3I0xuY/s329SZ9bdbbE2O2ZByu+t27yuGtPGXW7Oa28wHq39DrKEGpbpcfVTjbzLr71SUSued9pihkyU4H3bsEpGepRD1wmOegQMA2IUBpzE+W8PVwis7gtf0DZY5M6J9oZOTQVuc2qs5gvBaKWi58xCuxABZBImB2+Xt+4hd2yB0yPQ0YUU5tLWcB2I923nvharlVQmV/YkxZLvwnqB2MhR+BescbzOSW0k0lZPqnhU26HJF4+kZJYtBboH96fJL87Xi3kTPLAwQWGrqgbsIDmfqngvzVGVRrg4mipm0RmiDGRYYnk3EpVPga9Cg5K5OO3EqHxT8BNTcPp2w1Rz4RhY4CNWay4p3FGRTSAGiD3/HMgldZMTdYXj126fn8lXX/MHOZVljqvAmqqMvjFIa37tBROiUswU/X4ozKjB6wgiocpXIGnGXKLUfPCowlceU2lSty/quWUtiA/wzgbAUc/jihfdTazihUicr5kvoXtwunHjMt+lqN1Ph9vumOiZVBOoNGcnSSZ3vXSNXOu4yYR7CVceMnDdza66qlhtJGfh9UVeLvd3tmNhpOQI5w3GFQ5OalA1+Bu7GvY52THsh4kV5DGffCLGHCjK+uTdGIyPWfkezCF0vFOL8rvf+AM1PveFGJMyEw1KTQeMacdh4IsuwMf0I6k+5y1RTp3t/68HpTk6Bg80cmBay5a8e2oEe4IUaXvupoK3v1IQMP6jTWeK1iWNT27ciPfwe8s+Lse0yZ4p9GKYKSWN53d4eicAE10GhFvOv7rR1gGhu5vIwsTmO5tLmFDl9RP7q1l76lveHkP7eHEZd5du3pd5ZNptlamew/TEk/QhMxxLEXFTKQxrh1HmJbfuka7I/A3VijfoN+Kk3WWmNTMRMVzkjX83n7ssTc4tbt3jtVA5X+UWYGvUMWdiIgpLj4iTUoleus4R8n7EGZHAAGMthkJX+EGRwcpdjMBIKvuaIpdJHLsb0WKXHRIpN2wAbL9vUGNSkBGV3ojYvplgZ6cp74fNxjlrg+n/GARCj+p2qrovaadZd3ipQR8Ict5ogeTylzP+cWYRX6ot7+yqsRiopChRMspYzAqrLDX/nC+YyJDwb4rNfHTIwKgVKLkHVCEpzXEldNshJewIvRdxpbXo6RF9vWHek37YePkAPYqzy/BO0G7uDfvlA4zwnbxLCuaZPXGteeapvfxFfWlO7OjFBbV2IrxULKKV2nr7ZQglo5TQCflXqgOzkOGz0GNYG4PzRnIVTP8M8rSsuaMRXan1dTnmelaVlYMO2VbvKntToYxXRo0OY7o0iUQ8IITDfv6g+WwSez6/zLc3qYuUUe+emxYfVcr5bR+J+jRqZHtuBouenAd51bqFTfMGBYnbabwUNCQf3cOIfbHeiBQSqdFhs5DwXTAaDkyplq2SpaOaaRMMQ3E7UDTGFlGIxt4V4QJWpeur52FJisvEmRv2mZeBcICSckdeC9qj6IzWkL2wvPMu3BYS6d5QuA3GZLm9Jf5B/YdTYltCy03v+6hmyraEZ+NbX84FVs1l5dT6Ddt3+BBv67rlom3xUEW0WIVwu1YdcwvjWimEAyc9zORcwTOTw+qjIwtIMrTHMeCoFF6PPQaV2FlAF127XozJKQ544mBPPOlO14BASZUnLjg+wNcfxZ6Ebgg7E3WE1QcJllMytEdNE7SiPYpEE4q449/mLFgu60pBRTwamPY0YeM0eCwkrIhXJEMHIoG2mD9+R92zIdk5xolT/dskzkJoaKjjZn+Gcj700PBnikGhfCxc0Rzya2HWLbQxZ44G1jPpiC3a6M21q80HJ0EviLGQqzSiWDMstlIDPeJXvfFYX7Ym7KnTG1/e4gzTcuj0ay6CiS1dWGNOT+If7Kz1aKuCpL6LAHShvQMhOh3Jc33WTEEZ49G6T08NArAx6IjjIe4Blh4DEgQtClpMXl+MorvwASVQX7jLGUIDACl1CYKkWgF9UgihKlAhzkMYz5aXN263RIxasRQdG+rz4PR788RgMV3O7MHd7XK1VAnEauV7XkRi8CEsPacuoNePqc2MycswPMu7cBX0P3yTerUmr0PXLj+XDk3t8nMM+v4f6izM865WQDJQm8JP0BOglIWjKtp7BvxegRFm+zGjXKUgDXI+bSo4u+ttsdORTZvobWOiPV48A2YwmKkpxNOZKcG717RscKKJKCinTQU2tn2x5f5AcUYQowjIwesD0S4ovFMYeFfAUUkzk0ojJnzStVLOkvQyfvnn6C9Hor/MiF9+HP/lJ/FfHjRklI10XrDI1AKuOMhK+mCdSAPlE4loAVAe9wcXxfh51wuqmTmybPDpBAwZLVQSsM5pNUFdB4i7XatgKaQ6n6D6VUw6kJp3MNsqHoOd+0IFV57kWYmHjTsdqyqbmF0yJN1ZQW51zCFMiwkoZ2ZC5EpVXtTYkXxxMGRs9fq+wp5TjlS/VfyMSRg1QbfUXW2zxQxAvC22OlDxPy9aoIYM7lhsJGOE9SE0pF+Nr4abW2ICohADIUcy0PG0gZHabth8m98qm3LSiZUrsfHt6bnLl1qkfImPDRBuLW2FW5zEuJiL4TJH+sopJhtAjBSPmXyHQ8jwDQba29I5HW7JmShL6gtkCpohuUsXK7V4rltUrce/jRn2QnhPQWoEp2mqYIFVVgp47RvAloSwzZWnzbUz9vJ1jDSngcHz9Cvfw8v6/C1cgNLIgbbmuHR9b439QnhTHGijGz/ULy/7zM+Wow04gipbvySK44cH+x6nCObwzKdHBjNpYcVFyBpwtQFucQ1INcC/pfjkB7n7GpERFnJYnVvEjHEhEzWPC5mZYlyeEcQftEuKP/iCknGBqaamLvBBLXxPLmK8ZmFcM47gPVs98JJ2waUtA8/6mUYLnykEtrF6q7kIn/0hLsZ4NBpt9//xdv+MG+5csJTC1QKDyaI1/j5x0ZB7nE7Uoy/WB9ONtASLGFoyoxZAWlRaMKyWSABD8wFicCBCjKjoJqgEI19LjJGNXuMYQ8rjKq/QKijJaEk3lR5mGBFjNWenRZvPKnFNpayqkSaNcFfC2lV143TX6OFx/fo6kkyG0JPbqdNkWV25JsYYeLf2rjsiuqGNbrxyxlXKTkn3lVvQ8J5mUvMkC4wr4zhoGuC+Mr+hmeA+8mRwn2k+yEfsNv3gHqgiBf7cxq7HnO5DQ6qsFnyCL0znQ3AACTeGowM/jbB+IyoSlkddHiNiV0WX6x/voGZUKV3b4D8dVT+VwP1LaY7TmXm6HAUg1Zx99gwI+KA8DJs/cPr/6V75f/e+C/ejza5Ly27H7v5PPk0d7N93qH/fR/36qBkJAxa8v+717mK5B5WBM+RRZfglRbnnzDK+ZsEujy17mMKI4n8CrDkQ1PZ5NE/q586AYhP0wVOWocrRCNoL8nhIPbs8etW3rWW1+xQ8i1lpMLM89bZOTv4uf5ZLwKAlDlJ3qNN3LYa0N+meTZGVWgKUmFz66X0PodySDgh1MjKA6qyC0SKkAJkAkndCty+pSgZvEvnKe/iH22d1TEFwCIQiIwGOioOkAyjuTYIYmoiZTepXnpHsw2FgBwwo/NbISBl9h1j44++QJ1X0HX0PEJ81aTWQAmo94DKSBtHpkzQe9kMusmCtTh5nlvg44dlJmXCzp9HeOMeMOVFK89OwzKmfmmwsTDemX9QXZ+yFK43vHsgN8DNP11+er8/c37h4rStoh1G/J82ymOwssTiz5a7ykSpGvezHp3IoY1pDwCBYM5FKZYpDqVREsHiomOUUJQdNOJUPpEd3uxU+NvOl36micoTxFoRUWraBnObYjhGZGdVKkEfQieHBg82cjTHhJh0HjgLgCYPtRdwBpYy3winQrjl59YQooOxMGNscSM7tFRwbh7fVnL5m37u+/vzx+pubuO2z8gVwmK+daDaP5+B7xF9GzcKO2HvxnYNGh56iKexp299lFMVahwdk6ozgeGiJOAlElq2tuLsoM6R2yLFpHojbAIV3RihgMuKmxkzqaGKJjkhqqTMwq1w2KzCncQ8mcsfUYj0KFl0r/eo2p8+DxTvOaC2lsdsdbLFdZxBP3OtOcHCXxLLXYZq5x7gqmp4VBRk80avuCSfvPggaWPg2l05u3Dgt3v/1di403bY5EoOGxqbapTgOe9GixZ760p467bJry+05rcCd64tcUJzgiW9HpNO5gHDFOtYzhBdHnet4I1+nbjj3MEFLaUpAlzCsSrEMJly56jaPiYyCrmzi65gYLSBc22bUXTBafyh2l66RoSuaZc4qHPqLL9rCSZdyUUzWpMDItFB45fjqy+YiXgENEBrnl+y7p+y5qxtn5kSbLFQBUsIJiPVwksJ3AuYWLz7hB144ItPiSN7kNNwwHfESSy0bJF4BMvNoY6L9EOj+QNkwpzWRTatJcN1Jmix6ccjVdQuY2aYtcBU/2dojdGDjQXywsm6DzOBI3VZhJ96nq+bpkrK+9n2QiZ+C8LdnLznwJRkXpqFRLQK9dXaBzg+czzscqBQtxnv0MKsBHh4gx7TJGEHuzrYdbdsAfKOwkNHnsHSCu2xPDjmZ5uDeCxeqKtgONsW8dZ5SktkpwgtnwOp5+94PntDrALkAK6HOUDGxwtwX9ZsX6pev8V3KG599Y8/dX39zFv6lg9xnZaAdEap9U+UTUfT+v6013gmVkg5YnxQTX7qu3TN0Eq2UmScbXz5p2yCvJTu3V5+Z8Q0J97E+PYEOTfJdomds6gIIFPvVbRDwsj15E0SJU3ljw5baMFHLNEZAT49FzuHWwFmf/9PNccZw1DMvNdH0cJeZ+ESOxYHAcGXpRKOk3Z0KuIcHPc5mbiwoq55rcqkLITbmJ3A7k251Y7QwB5drD015LkXZuD63/mYNU3ZQNk979imYHMHZQI29n4iDlWomV0SIjUvnNm7exQCRq29A16pUfjncGchlc2bZVU6gVQaL5BMMjeACF9e0HPUygn4mSUnyL9MuliI2LmnxbWXYeUuGTdI7vDh3vCmmnVawHrfuu3Kx1fEQcWu1MIM+FNxk+UQK5pDOUm3TzlGEjQosYJ12zp6ctd9cYR1Lk5a0//SsZunr1+Kp+yHdS5q72t4S6E24CKsLupWWW3CPXwT3okSjJHfktZDmPu8OY2vKAQZPN78QwnRVjD/odQoWecAydJ5rDJpSsbDNH27hazqNNqbH27pl9JzKmxpgd9/dXj4tk9OOBcXiK0BO7H2tTSrcrKGH3G7Kt0S8FJSh4XDBiSthNzcQx5eHA29L7E26uTiABrrfRCpJKZOvewKPuBXlCWLqIKy+ZDIo8JW+BQe/+vM8urdEeWNsMTH3YLGYD+mpTZEmb08cGnx/3uGCvyTyRjfuk9D+AGWqgGY5O0V9/rF8XTOC5+5beabtiR81BTsICpmfg0Fifo4+vycn3Fn1dHXlMI8+R/ESoVMp9DmkUoZya6IDouv/ABfxh08="},
    "rescore_offline.py": {"sha256": "beb4cc0b644f2467d873f80f552c68b63e03219bf7193511c03e254a1f8811c5", "bytes": 4808,
        "zlib_base64": "eNqdWO1z1MYZ/35/xVbtTKRyJ8dO0qaaOB2GupO+hGYCQ6dzvdEIac+3tU66rnRnm7NnDKXYgLGdhGAwBOIUAsMEDJkUDD7w/9KcdOdP/hf6PLuSTmfzodP7cKddPe8vv332FEXpXX3Q3zof//ty//xGfOu76NUX0cUrvY3z/fv/im+v9e+fj5Zu7HeWe9/u9F7u7i1eia89iZbuRksX9jtL3e0rcqf/5B8gJ36wGd2+vN+5+OPCuUIh3lqNv9vs7671N5dB3t7CWvzVZrRyJ7p/ub97Nbp5O3p0Pbr1gPwt8D2X7C0uA1deaXz1Rfflhe7OvWhtBQ0AO3dvdLcfSju7Oyt5WYXEDsFPSqNkhIwCU7TViRZfdrdX8i72bm5Ha8vx152os5oZRo6dOPXjwtm9qze6ry/jgkSfL8dPF+OFHfL7E386Llwi8GnMhjXfI5wGts+p6VerLvOo3pgl/ddfgLLezcfRqy+Jro/UaciZHYxwa3rk57pwU588UyiUSH9rh8gNEi9dI9m7/c5NsKy7vUCYM0NQ2MqT6O4GhDFefxatXotvLUAw4D0YDN71t57HT89Fd5/2f7gHjsAmKIec9B9viuSsxNce9c696O086j//TKQU4yrdT7JUIvH62f7rF/GtixBO6S/wfhCtXYkerX9oBs163eKzwkJgATPi73chFdE/n3W3H6FMkd/u7mZ8dgsDvvoQMtF7vN7buQNmJCq+3wQDosfLGT2oiJaeQHpB4N76D93tS3s31kB+98WdbmdDZg2kSXugNIjt1xsuDalDutsvSdVyA4qyFUUpsHrD5yGxg1b6OHmGNdLnmhXUXHY6Xcof2NCbIXPTXfQufeY0fQpmg0KV+3VQ7rrUDpnvBYkAcsxveiHl8n3DClFH+u4TWBYKH018OkHGxUI1zSpzqWlqOpSN77aoqukNi1MvLEycOvpH88SxT3/3yUmgFkwjRAmCcMxkXhDyptBr0pblQo0phaBBbSAc9kPHXRNtkYpc37aQTX2zIKVIcmq1Am0dllj3nSZIEjJRuopfmlCvu77lUK7TGVAq6VTa0gqF40c/Rpc51TFdYIjKFRViZ9WtOYfSRkDp1JzfoJ7FNFPVj/wavk9bAbPnbD+c4z7ogwfNDGvMmyqpvjcH3QWcVavpwrYi+m/4Awos153z/uocAWHlt0u/skrVSnt0bF77maIVPp44edT8w8RfToBZqsKbHriuNLjfYuAAPoP5FOOhgB3UbHLxXGXeJOUNzrwQl44VWuaBvdDikzQ0Qd6wUUpWqImmekPSo0upMKHYmjFDf4p6gXhN6w3KrbDJaWIAC2rUMa0QnCgUIAQQVcsxOQXQcQIVK84QtaWR0ofEZUFYdpgdVgxhDsaYcvAZO0HHFWFVUaZ60KxW2QwZHycKAI5CKHSSoBeMju9RYGvPi9U0C2uJLKERLOPoDfVs34F4jCvNsFp6X9GIFZCqkUWi6nOCoEiYl9/GD5gh4BIKkjVUzTiUUi7KGxtSlFmgIrl2iIxhQGwdcqAqlHOfgxEsIMfBfuMNZSI9KwNLWQFoVSoVUad2QaqEuHukLEhYRZjP0PYAGoI6Ku5rlSQNAaQKQP9gBsAfqdez6hhBEWt8LqQBSQIPYlUReQimxH0lF4WEG38AKup+i0o2Vf5oeXuF9LQ2xGGUM6pI/GZoOoynS4CbgHIzqFkGGiuMxoqRutEr0Jt3TurCXWxrvW6Fdk1FAvkCEuD5IakPbOcWg1I6ZblNOoEZUatKfG2xu/MMzhI4lOLrW0OH8Mb56LNX8Eqew/udjXYWs3lF6kgbtUhEmxaJ7KciEd1UJAE7Q4uk2gAb6/ok95uNQM2sS5lFqUsQUgxCfkoyo+BciS9dIn+fpt47+nvGu6cJGnXh82j1eXzlm3hpDc4nUsr8EzagKvwtG0ZptAI5ariWTVWlhPk04GtUk6+STNmBKLQDrSuNtGsWYAoSJKdJYjxWC8dCQfZBfH3XQVllQBVAl8og7ulOUTxinhUsb9rSxcJ0rdPUVeGd7YMWwK+KlmcGs0BRKHgyWRg1WAjWnK7E5DKgulMcUGsVcmRcGPiTgxZKoATR7/9yDNOCORMpAdhO0AcgVcXt8qiRWAaTk4UwJBDbEGWZx20jVxkJghtpiaSYa2TFkmCvkVTNEDoMcNxILC3mIdwgLvVUTIOGNicUwxJyQG5gzK1m6JuDTXWoZrUDWD/gyO2qklRisM/ZJPMsN8UUxGQTm0T0IjlClKEZTcnqP2WEYxpOhyCPtVNwFg+DbEYsSjWkM6F6COQHRZOmZ8oQospTEjOnsGYHJy7YIHaQZD5La1lJwMpJR2dReCd5k+ZoUrSy4ZQIEbTG3vuFoBvAmMQt6bqsdrmAQhIZKwpR0ugEC/X6FHyrcvIKxlEnHGYYHtOfEktJDrKmOQthGglaasILU1lVaWPM53XYhk4XZfG/0JvigAoStjIfbvAUSfOdmLTBm2QNJzvRKxKWpUfk1WnWAQ0TajyyA6gs0wpsxsZ/i+NzEQxwIA7jY9obTnR5ltFpF2MLUtSWsFr2PewDTws9SABBByvqUGOi8AAGkqOYWJ6D1LhGH3FPirZcwDvq/L/Ch4WJqUxNQ2SQdta18yNt2bTzRF6ZosULvc3HvZVF0k5iU37Lsu0mt+zZtyqG/m51Hu8d61/HXy7lSJgHYzNz3qrg2/wcCmfc0kO4ZuVoYdT2YPymkvpmdiuFKw1pi5BmxsCN69bFvW/wVITbbrR6nbSTwAgSZei8P4CGOJXkASobSvNQlrqGbIl9g73KASRLnMzTplsVlJ76lScYbB6UJjw1Rb8ih1giuIqcOqZoO6g4eJW6nAwzdYt5qphPBtMcjo3gKNzIdHCtpZEPyDsDPMNtkBfCPcvxbbhm5Zs+vYOlvOXRpLsGUAIkyTVRl1Cj5q5HEhJPz4YUSlDTa3TGYVCUYXJYc38aT/ByOoIJXQ0tm7/yk5cmyrwhRsvUmjGjUhketnNdr2T/MrQobzE6nWCIMv2GKVy0DgLquHJoIp8GE4FV/w1MfH9GyIDpDCYnRl0Hz5JgHG8QKvpSfruSQ/ppCTA1ihc+9dA+Mgiu4T4c+hfnq3v9xYeknY+omPH2O0snPjpagmiT9iBEZWN0rDL/n4Vv9zsXxdUHMm+K8840xdhgmlgfpqlI52SxFP4LiiEIDA=="},
    "sst2_validation.tsv": {"sha256": "a7a3bfee7fc80566578791dfea290715934cc82f436c893361c4d0108a3554f8", "bytes": 99186,
        "zlib_base64": "..."},
    "sst2_validation.metadata.json": {"sha256": "d9c018c30511f87199684de5b848a8037e968412ec101db82677663228a4e1ef", "bytes": 566,
        "zlib_base64": "eNptUstugzAQvEfKP0RcW8L6gTF8R+/IGBssOTbCkLaK8u811JWI1Jtndmdn1trH+XS5ZL1YRFBL1lwy973mN2uLwa4qe9+r0jtthq0YwoITGSZrdsFdWBP1xrtUcZHlFf4FVnTKttKvbgmRf2xkpFF8U0rfE4QNYr6h50F3VMAeTg3R6Z6CpTnZ5IPZ2YNexVjrnqr9d1SOjiPQQTnqVhs3qHmajds3BIU7ACCko5zVRP/9wChwybYGUQnSaaUqLTmUjJUVr2rUayVwDRUqa0Kl5FhTwiSvCWFI0h4QcEHKkmqeBvb+01kv+la5fvLJfFyWKTRFMer8ZubZz1fpb0mgviY/L6pvxd6KAbMcQY7wByINkIbyK2YVovAG0AAkldfaSCNsG/w6S/Visg5D3F0LqaJNkc4iFC9HcT49fwDBxJUo"}
}

DESTINATIONS = {"sst2_instruction_eval.py": EVAL_SCRIPT, "rescore_offline.py": RESCORE_SCRIPT,
                "sst2_validation.tsv": DATA,
                "sst2_validation.metadata.json": DATA_DIR / "sst2_validation.metadata.json"}
for name, item in EMBEDDED_ASSETS.items():
    raw = zlib.decompress(base64.b64decode(item["zlib_base64"]))
    if len(raw) != item["bytes"] or hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise RuntimeError(f"内置文件 {name} 校验失败，Notebook 可能被改动过。")
    DESTINATIONS[name].write_bytes(raw)
save_json(META_DIR / "embedded_assets.json",
          {name: {k: v for k, v in item.items() if k != "zlib_base64"} for name, item in EMBEDDED_ASSETS.items()})

# 直接调用评测脚本里的函数算实验指纹和文件名，保证和评测脚本自己算的一致
_spec = importlib.util.spec_from_file_location("sst2_instruction_eval", EVAL_SCRIPT)
EV = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(EV)
ROWS = {0: EV.load_rows(types.SimpleNamespace(data_file=str(DATA), n=0))[0],
        N_SUBSET: EV.load_rows(types.SimpleNamespace(data_file=str(DATA), n=N_SUBSET))[0]}
if len(ROWS[0]) != 872 or sum(r["label"] == 1 for r in ROWS[0]) != 444:
    raise RuntimeError("验证集条数或标签数量不对。")


def run_file(model, prompt, think, n, url):
    """评测脚本会用的断点文件名（不含 .jsonl）和完整的实验指纹。"""
    fingerprint, _ = EV.experiment_fingerprint("ollama", model, prompt, think, url,
                                               EV.auto_max_tokens(prompt, think),
                                               EV.auto_temperature(think), ROWS[n])
    return EV.run_name("ollama", model, prompt, think, n) + "_" + fingerprint[:12], fingerprint


def model_slug(model, digest):
    """结果文件夹名：模型名加版本前 12 位，和上次一样。"""
    return re.sub(r"[^A-Za-z0-9._-]+", "-", model) + "_" + re.sub(r"[^A-Za-z0-9]", "", digest.removeprefix("sha256:"))[:12]


positives = sum(r["label"] == 1 for r in ROWS[N_SUBSET])
print(f"验证集 872 条（正面 444、负面 428）；前 {N_SUBSET} 条正面 {positives}、负面 {N_SUBSET - positives}。")
for name, item in EMBEDDED_ASSETS.items():
    print(f"  {name}: {item['bytes']:,} 字节，SHA-256 {item['sha256'][:16]}…")

In [ ]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("没有发现 NVIDIA GPU。请在 Settings 里把 Accelerator 设为 GPU T4 ×2 后重新运行。")
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
gpu_lines = [line for line in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines()
             if line.startswith("GPU ")]
if smi.returncode != 0 or not gpu_lines:
    raise RuntimeError("nvidia-smi 运行失败：\n" + smi.stdout + smi.stderr)
(META_DIR / "nvidia_smi_start.txt").write_text(smi.stdout, encoding="utf-8")
print("\n".join(gpu_lines))
if len(gpu_lines) < 2:
    note("只有一张 GPU：所有组都在 GPU 0 上依次跑，用时会长很多。")

LOCAL_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def version_matches(text):
    return re.search(r"(?<![0-9.])" + re.escape(OLLAMA_VERSION) + r"(?![0-9.])", text or "") is not None


def install_ollama():
    missing_tools = [name for name in ("curl", "zstd") if not shutil.which(name)]
    privilege_prefix = []
    if os.geteuid() != 0:
        if not shutil.which("sudo"):
            raise RuntimeError("安装 Ollama 需要 root 或 sudo 权限。")
        privilege_prefix = ["sudo"]
    if missing_tools:
        if not shutil.which("apt-get"):
            raise RuntimeError("缺少工具且找不到 apt-get：" + ", ".join(missing_tools))
        apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
        if subprocess.run(privilege_prefix + ["apt-get", "update", "-qq"], env=apt_env).returncode != 0:
            print("apt-get update 有报错，继续尝试安装：", ", ".join(missing_tools))
        subprocess.run(privilege_prefix + ["apt-get", "install", "-y", "-qq"] + missing_tools, env=apt_env, check=True)
        still_missing = [name for name in missing_tools if not shutil.which(name)]
        if still_missing:
            raise RuntimeError("安装后仍缺少：" + ", ".join(still_missing))
    installed = ""
    if shutil.which("ollama"):
        existing = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
        installed = existing.stdout + existing.stderr
    if not version_matches(installed):
        installer = Path("/tmp/sst2_ollama_install.sh")
        subprocess.run(["curl", "-fSL", "--retry", "3", "https://ollama.com/install.sh", "-o", str(installer)],
                       check=True)
        save_json(META_DIR / "ollama_install.json",
                  {"url": "https://ollama.com/install.sh", "requested_version": OLLAMA_VERSION,
                   "sha256": hashlib.sha256(installer.read_bytes()).hexdigest(),
                   "downloaded_at_utc": datetime.now(timezone.utc).isoformat()})
        subprocess.run(privilege_prefix + ["env", "OLLAMA_VERSION=" + OLLAMA_VERSION, "sh", str(installer)],
                       env=dict(os.environ, OLLAMA_VERSION=OLLAMA_VERSION), check=True)
    check = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    cli_version = (check.stdout + check.stderr).strip()
    if not version_matches(cli_version):
        raise RuntimeError("Ollama 客户端版本不是 " + OLLAMA_VERSION + "：" + cli_version)
    return cli_version


def parse_load_block(text):
    """从服务日志里最近一次启动 llama-server 之后的部分，读出实际的并行数、上下文和 GPU 层数。"""
    start = text.rfind("starting llama-server")
    if start < 0:
        return None
    block = text[start:]
    command = block.split("\n", 1)[0]
    c = re.search(r" -c (\d+)", command)
    np_ = re.search(r" -np (\d+)", command)
    ctx_seq = [int(x) for x in re.findall(r"n_ctx_seq\s*=\s*(\d+)", block)]
    seq_max = [int(x) for x in re.findall(r"n_seq_max\s*=\s*(\d+)", block)]
    offloaded = [(int(a), int(b)) for a, b in re.findall(r"offloaded (\d+)/(\d+) layers to GPU", block)]
    return {"command_c": int(c.group(1)) if c else None, "command_np": int(np_.group(1)) if np_ else None,
            "n_ctx_seq": min(ctx_seq) if ctx_seq else None, "n_seq_max": max(seq_max) if seq_max else None,
            "offloaded": offloaded, "all_on_gpu": all(a == b for a, b in offloaded) if offloaded else None}


class OllamaServer:
    def __init__(self, name, gpu, port):
        self.name, self.gpu, self.port = name, gpu, port
        self.url = f"http://127.0.0.1:{port}"
        self.log_path = META_DIR / f"ollama_server_{name}.log"
        self.process, self.parallel, self.context = None, None, None
        self.single_only = set()  # 并行模式下出过问题的模型，只用单路跑
        self.checked = {}         # (模型, 并行数, 上下文) -> 加载后从日志读到的设置（服务重启后清空）
        self.parallel_context = {}  # 并行数 -> 试过可行的 OLLAMA_CONTEXT_LENGTH
        self.history = []         # 每次启动服务的设置
        self.load_log = []        # 每次加载模型后的检查结果

    def env(self, parallel, context):
        env = dict(os.environ)
        env.update(OLLAMA_HOST=f"127.0.0.1:{self.port}", OLLAMA_MODELS=str(MODEL_DIR),
                   OLLAMA_NUM_PARALLEL=str(parallel), OLLAMA_CONTEXT_LENGTH=str(context),
                   OLLAMA_KEEP_ALIVE="30m", OLLAMA_MAX_LOADED_MODELS="1", OLLAMA_NOPRUNE="1",
                   CUDA_VISIBLE_DEVICES=self.gpu)
        return env

    def api(self, endpoint, payload=None, timeout=10):
        body = None if payload is None else json.dumps(payload).encode("utf-8")
        request = urllib.request.Request(self.url + endpoint, data=body, headers={"Content-Type": "application/json"})
        with LOCAL_OPENER.open(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))

    def stop(self):
        if self.process is not None and self.process.poll() is None:
            for sig, wait in ((signal.SIGTERM, 30), (signal.SIGKILL, 30)):
                try:
                    os.killpg(self.process.pid, sig)
                except ProcessLookupError:
                    break
                try:
                    self.process.wait(timeout=wait)
                    break
                except subprocess.TimeoutExpired:
                    continue
        self.process = None
        deadline = time.monotonic() + 30
        while time.monotonic() < deadline:  # 等端口真正空出来
            try:
                self.api("/api/version", timeout=1)
                time.sleep(1)
            except (urllib.error.URLError, TimeoutError, OSError):
                break

    def start(self, parallel, context):
        if self.process is not None and self.process.poll() is None and (self.parallel, self.context) == (parallel, context):
            return
        self.stop()
        with self.log_path.open("a", encoding="utf-8") as log:
            log.write(f"\n===== {datetime.now(timezone.utc).isoformat()} 启动服务：GPU {self.gpu}，"
                      f"OLLAMA_NUM_PARALLEL={parallel}，OLLAMA_CONTEXT_LENGTH={context}\n")
        log = self.log_path.open("a", encoding="utf-8")
        self.process = subprocess.Popen(["ollama", "serve"], env=self.env(parallel, context), stdout=log,
                                        stderr=subprocess.STDOUT, start_new_session=True)
        deadline = time.monotonic() + 120
        while True:
            if self.process.poll() is not None:
                raise RuntimeError(f"{self.name} 的 Ollama 服务启动失败：\n" + tail(self.log_path, 2000))
            try:
                version = self.api("/api/version", timeout=2).get("version")
                break
            except (urllib.error.URLError, TimeoutError, OSError):
                if time.monotonic() >= deadline:
                    raise RuntimeError(f"等待 {self.name} 的 Ollama 服务超时，见 {self.log_path.name}") from None
                time.sleep(1)
        if version != OLLAMA_VERSION:
            raise RuntimeError(f"{self.name} 的服务版本 {version} 不是 {OLLAMA_VERSION}")
        self.parallel, self.context = parallel, context
        self.checked = {}
        self.history.append({"at_hours": round(hours_used(), 3), "parallel": parallel, "context": context})

    def check_model(self, model):
        """加载模型（空提示词只加载、不生成），再从日志读出实际设置。"""
        key = (model, self.parallel, self.context)
        if key in self.checked:
            return self.checked[key]
        offset = self.log_path.stat().st_size if self.log_path.exists() else 0
        info = {"model": model, "parallel": self.parallel, "context": self.context}
        try:
            self.api("/api/generate", {"model": model, "prompt": "", "stream": False, "keep_alive": "30m"}, timeout=900)
            time.sleep(2)
            text = self.log_path.read_bytes()[offset:].decode("utf-8", "replace")
            parsed = parse_load_block(text)
            info.update(parsed or {"note": "日志里没有新的加载记录（模型已在内存里或日志格式不同）"})
        except Exception as e:
            info["error"] = f"{type(e).__name__}: {' '.join(str(e).split())[:300]}"
        self.checked[key] = info
        self.load_log.append(dict(info, at_hours=round(hours_used(), 3)))
        return info

    def ensure(self, model, think):
        """按这一组的需要设置服务并加载模型，返回同时发几个请求。不思考的组和出过问题的模型用单路。"""
        want = 1 if think == "off" or PARALLEL <= 1 or model in self.single_only else PARALLEL
        if want > 1:
            info = None
            known = self.parallel_context.get(want)
            contexts = [known] if known else []
            contexts += [c for c in (CONTEXT_PER_REQUEST, CONTEXT_PER_REQUEST * want) if c != known]
            for context in contexts:
                self.start(want, context)
                info = self.check_model(model)
                usable = not info.get("error") and info.get("all_on_gpu") is not False
                ctx = info.get("n_ctx_seq")
                if usable and ctx is not None and ctx >= CONTEXT_PER_REQUEST:
                    self.parallel_context[want] = context
                    slots = info.get("n_seq_max") or want  # Ollama 可能自己调低并行数
                    if slots < want:
                        note(f"[{self.name}] {model}：Ollama 只开了 {slots} 路并行（设置的是 {want} 路）。")
                    return max(1, min(want, slots))
                if not (usable and ctx is not None):  # 不是上下文不够的问题，加大上下文也没用
                    break
            self.single_only.add(model)
            note(f"[{self.name}] {model} 不能按 {want} 路并行运行，改为单路。检查结果：{info}")
        self.start(1, CONTEXT_PER_REQUEST)
        info = self.check_model(model)
        if info.get("error"):
            raise RuntimeError(f"{model} 在 {self.name} 上加载失败：{info['error']}")
        if info.get("all_on_gpu") is False:
            note(f"[{self.name}] {model} 有部分层不在 GPU 上：{info.get('offloaded')}，会比较慢。")
        if (info.get("n_ctx_seq") or CONTEXT_PER_REQUEST) < CONTEXT_PER_REQUEST:
            note(f"[{self.name}] {model} 每个请求的上下文只有 {info.get('n_ctx_seq')}，比上次少。")
        return 1


OLLAMA_READY = False
SERVERS = []
try:
    try:
        subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)  # 同一会话重跑时关掉旧服务
        time.sleep(2)
    except FileNotFoundError:
        pass
    CLI_VERSION = install_ollama()
    for index, line in enumerate(gpu_lines[:2]):
        name = f"gpu{index}"
        server = OllamaServer(name, str(index), SERVER_PORTS[name])
        try:
            server.start(1, CONTEXT_PER_REQUEST)
        except Exception as e:
            if index == 0:
                raise
            note(f"GPU {index} 上的服务没能启动，只用 GPU 0：{type(e).__name__}: {e}")
            server.stop()
            continue
        SERVERS.append(server)
    save_json(META_DIR / "ollama_servers.json",
              {"cli_version": CLI_VERSION, "python": sys.version, "platform": platform.platform(),
               "models_directory": str(MODEL_DIR), "parallel_for_thinking": PARALLEL,
               "servers": [{"name": s.name, "gpu": s.gpu, "url": s.url} for s in SERVERS],
               "checked_at_utc": datetime.now(timezone.utc).isoformat()})
    OLLAMA_READY = True
    print(f"Ollama {OLLAMA_VERSION} 已就绪：" + "，".join(f"{s.name} → {s.url}（GPU {s.gpu}）" for s in SERVERS))
except Exception as e:
    note(f"Ollama 安装或启动失败，无法运行实验：{type(e).__name__}: {e}")

In [ ]:
MODEL_INFO = {}


def pull_models():
    first = SERVERS[0]
    print(f"模型目录剩余空间 {shutil.disk_usage(MODEL_DIR).free / 2 ** 30:.1f} GB")
    for model in OLLAMA_MODELS:
        try:
            started = time.monotonic()
            pull = subprocess.run(["ollama", "pull", model], env=first.env(first.parallel, first.context),
                                  capture_output=True, text=True, timeout=3600)
            if pull.returncode != 0:
                raise RuntimeError((pull.stdout + pull.stderr)[-800:])
            entry = next((item for item in first.api("/api/tags").get("models", [])
                          if model in (item.get("name"), item.get("model"))), None)
            if entry is None or not entry.get("digest"):
                raise RuntimeError("下载后在 /api/tags 里找不到这个模型的 digest")
            digest = entry["digest"].removeprefix("sha256:")
            show = first.api("/api/show", {"model": model}, timeout=60)
            for other in SERVERS[1:]:  # 两个服务共用模型目录；另一个服务看不到时再通过它下载一次
                try:
                    other.api("/api/show", {"model": model}, timeout=60)
                except urllib.error.HTTPError:
                    subprocess.run(["ollama", "pull", model], env=other.env(other.parallel, other.context),
                                   capture_output=True, text=True, timeout=3600, check=True)
                    other.api("/api/show", {"model": model}, timeout=60)
            slug = model_slug(model, digest)
            results_dir = OLLAMA_RESULTS / slug
            results_dir.mkdir(parents=True, exist_ok=True)
            same = digest == EXPECTED_DIGESTS.get(model)
            MODEL_INFO[model] = {"digest": digest, "same_as_before": same, "slug": slug,
                                 "capabilities": show.get("capabilities", []), "details": show.get("details", {}),
                                 "size": entry.get("size"), "results_dir": str(results_dir),
                                 "pull_seconds": round(time.monotonic() - started, 1)}
            save_json(META_DIR / f"model_{slug}.json",
                      {"model": model, **MODEL_INFO[model], "tag_entry": entry,
                       "show": {k: v for k, v in show.items() if k not in ("modelfile", "license", "tensors")},
                       "recorded_at_utc": datetime.now(timezone.utc).isoformat()})
            print(f"{model}: 版本 {digest[:12]}（{'与上次相同' if same else '与上次不同'}），"
                  f"{show.get('details', {}).get('quantization_level')}，下载用了 {MODEL_INFO[model]['pull_seconds']:.0f} 秒")
            if not same:
                note(f"{model} 的版本 {digest[:12]} 与上次的 {EXPECTED_DIGESTS.get(model, '')[:12]} 不同："
                     "上次的断点不能接着用，这个模型的组从头跑；和上次的结果比较时要注意模型版本变了。")
        except Exception as e:
            note(f"{model} 下载失败，跳过它的实验：{type(e).__name__}: {' '.join(str(e).split())[-500:]}")
    save_json(META_DIR / "models.json", MODEL_INFO)


if OLLAMA_READY:
    try:
        pull_models()
    except Exception as e:
        note(f"下载模型时出错：{type(e).__name__}: {e}")

In [ ]:
CHECKPOINT_LOG = []
NAME_RE = re.compile(r"^ollama_(.+?)_(basic|cot|role_cot)_think-(on|off)_(all|n\d+)_([0-9a-f]{12})\.jsonl$")


def model_from_file_name(name):
    m = NAME_RE.match(name)
    return None if not m else m.group(1)[::-1].replace("-", ":", 1)[::-1]  # qwen3.5-4b -> qwen3.5:4b


def read_records(lines):
    records = {}
    for line in lines:
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue  # 被强行中断时最后一行可能不完整
        if isinstance(rec, dict) and isinstance(rec.get("idx"), int) and rec.get("error") is None:
            records.setdefault(rec["idx"], rec)
    return records


def merge_into(dest, lines, source):
    """把一批记录并进断点文件：同一条已经有了就保留原来的，请求失败的记录不要（会重新请求）。"""
    dest = Path(dest)
    existing = read_records(dest.read_text(encoding="utf-8").splitlines()) if dest.exists() else {}
    incoming = read_records(lines)
    added = {i: rec for i, rec in incoming.items() if i not in existing}
    if added or not dest.exists():
        merged = {**existing, **added}
        dest.parent.mkdir(parents=True, exist_ok=True)
        tmp = dest.with_suffix(".tmp")
        tmp.write_text("".join(json.dumps(merged[i], ensure_ascii=False) + "\n" for i in sorted(merged)),
                       encoding="utf-8")
        tmp.replace(dest)
    CHECKPOINT_LOG.append({"file": f"{dest.parent.name}/{dest.name}", "source": source,
                           "records_in_source": len(incoming), "added": len(added)})
    return len(added)


DEAD_SOURCES = set()  # 连不上的下载地址，后面的文件不再试


def download(name, item):
    errors = []
    for template in CHECKPOINT_URLS:
        if template in DEAD_SOURCES:
            continue
        url = template.format(commit=CHECKPOINT_COMMIT, name=name)
        for attempt in range(2):
            try:
                with urllib.request.urlopen(url, timeout=30) as response:
                    raw = response.read()
                if len(raw) == item["bytes"] and hashlib.sha256(raw).hexdigest() == item["sha256"]:
                    return raw, url
                errors.append(f"{url}：内容校验不通过（{len(raw)} 字节）")
                break
            except urllib.error.HTTPError as e:  # 地址能连上，但没有这个文件：换下一个地址
                errors.append(f"{url}：HTTP {e.code}")
                break
            except Exception as e:  # 网络问题：稍等再试一次
                errors.append(f"{url}：{type(e).__name__}: {e}")
                time.sleep(3)
        else:
            DEAD_SOURCES.add(template)
    raise RuntimeError("；".join(errors[-2:]) if errors else "所有下载地址都连不上")


def seed_from_github():
    failed = []
    for name, item in CHECKPOINTS.items():
        jsonl_name = name.removesuffix(".gz")
        model = model_from_file_name(jsonl_name)
        if model not in OLLAMA_MODELS:
            continue
        try:
            raw, url = download(name, item)
        except Exception as e:
            failed.append(name)
            print(f"断点 {name} 下载失败：{e}")
            continue
        # 放进上次那个模型版本的文件夹；版本没变时就是这次的结果文件夹
        dest = OLLAMA_RESULTS / model_slug(model, EXPECTED_DIGESTS[model]) / jsonl_name
        merge_into(dest, gzip.decompress(raw).decode("utf-8").splitlines(), url)
    if failed:
        note(f"有 {len(failed)} 个断点没能从 GitHub 下载，这些组从头跑（汇总表里也会缺上次已完成的组）："
             + "、".join(failed))


def seed_from_kaggle_input():
    if not KAGGLE_INPUT.is_dir():
        return
    for path in sorted(KAGGLE_INPUT.rglob("*")):
        try:
            if path.is_file() and NAME_RE.match(path.name) and path.parent.name.startswith("qwen"):
                merge_into(OLLAMA_RESULTS / path.parent.name / path.name,
                           path.read_text(encoding="utf-8", errors="replace").splitlines(), str(path))
            elif path.is_file() and path.suffix == ".zip" and path.name.startswith("sst2_"):
                with zipfile.ZipFile(path) as archive:
                    for member in archive.namelist():
                        parts = member.split("/")
                        if len(parts) >= 2 and NAME_RE.match(parts[-1]) and parts[-2].startswith("qwen"):
                            text = archive.read(member).decode("utf-8", errors="replace")
                            merge_into(OLLAMA_RESULTS / parts[-2] / parts[-1], text.splitlines(), f"{path}:{member}")
        except Exception as e:
            note(f"读取 {path} 里的断点时出错，跳过：{type(e).__name__}: {e}")


def count_usable(path, fingerprint, n):
    """和评测脚本的规则一样：指纹、句子和标签都对得上的条目才算。"""
    if not path.exists():
        return 0
    rows = {r["idx"]: r for r in ROWS[n]}
    recs = read_records(path.read_text(encoding="utf-8", errors="replace").splitlines())
    return sum(1 for i, rec in recs.items() if i in rows and rec.get("fingerprint") == fingerprint
               and rec.get("sentence") == rows[i]["sentence"] and rec.get("label") == rows[i]["label"])


ITEMS = []


def build_items():
    servers = {s.name: s for s in SERVERS}
    for exp in PLAN:
        info = MODEL_INFO.get(exp["model"])
        item = dict(exp, status="pending", server=None, pinned=None, done_before=0, minutes=0.0,
                    workers=None, attempts=[], est_seconds=0.0, error=None, note=None,
                    started_at_hours=None)
        ITEMS.append(item)
        if info is None:
            item["status"] = "skipped_model_unavailable"
            continue
        results_dir = Path(info["results_dir"])
        found = {}
        for name, port in SERVER_PORTS.items():
            file_name, fingerprint = run_file(exp["model"], exp["prompt"], exp["think"], exp["n"],
                                              f"http://127.0.0.1:{port}")
            found[name] = count_usable(results_dir / f"{file_name}.jsonl", fingerprint, exp["n"])
        best = max(found, key=found.get)
        if found[best] > 0:
            if best in servers:
                item["pinned"], item["done_before"] = best, found[best]
            else:
                note(f"{exp['model']} {exp['prompt']} 思考 {exp['think']}：已有 {found[best]} 条断点是 {best} 的服务跑的，"
                     "这次没有这张卡，只能从头跑。")
        rate = SECONDS_PER_ROW.get((exp["model"], exp["think"]), {}).get(exp["prompt"], 30.0)
        item["est_seconds"] = (exp["n"] - item["done_before"]) * rate
        if item["done_before"] >= exp["n"]:
            item["note"] = "断点里已经齐了，只需要生成汇总文件"


if OLLAMA_READY:
    try:
        seed_from_github()
        seed_from_kaggle_input()
        save_json(META_DIR / "checkpoints.json", CHECKPOINT_LOG)
        build_items()
    except Exception as e:
        import traceback
        note(f"准备断点时出错：{type(e).__name__}: {e}")
        print(traceback.format_exc())
    print(f"{'模型':<13} {'模板':<9} {'思考':<4} {'已有':>5}  在哪张卡上跑")
    for item in ITEMS:
        where = item["pinned"] or ("哪张卡空了就在哪张" if item["status"] == "pending" else item["status"])
        print(f"{item['model']:<13} {item['prompt']:<9} {item['think']:<4} {item['done_before']:>3}/{item['n']}  {where}")
    total = sum(item["est_seconds"] for item in ITEMS) / 3600
    print(f"\n按上次单路的速度估计，一共约 {total:.1f} 小时的工作量；两张卡、开思考时 {PARALLEL} 路并行，实际会少很多。")

In [ ]:
PLAN_LOCK = threading.Lock()
PRINT_LOCK = threading.Lock()
STOP = threading.Event()
RUNNING = {}  # 卡名 -> 正在运行的评测进程


def save_plan_status():
    try:
        with PLAN_LOCK:
            save_json(META_DIR / "plan_status.json", ITEMS)
    except Exception as e:  # 记录状态失败不影响实验
        print("保存 plan_status.json 失败：", e)


def say(prefix, text):
    with PRINT_LOCK:
        print(f"[{prefix}] {text}", end="" if text.endswith("\n") else "\n", flush=True)


def stop_process(process, reason, prefix):
    """让评测脚本像按了 Ctrl+C 一样退出：它会先保存断点。"""
    if process.poll() is None:
        say(prefix, f"[{reason}] 发送中断信号，已完成的条目会保留……")
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            return
        try:
            process.wait(timeout=90)
        except subprocess.TimeoutExpired:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()


def take_item(server):
    with PLAN_LOCK:
        pending = [it for it in ITEMS if it["status"] == "pending" and it["pinned"] in (None, server.name)]
        if not pending or STOP.is_set():
            return None
        if seconds_left() < MIN_START_MINUTES * 60:
            for it in pending:
                it["status"] = "skipped_budget"
            return None
        off = [it for it in pending if it["think"] == "off"]
        mine = [it for it in pending if it["pinned"] == server.name]
        if off:      # 先跑关思考的组，两张卡分着跑，大的先跑
            item = max(off, key=lambda it: it["est_seconds"])
        elif mine:   # 只能在这张卡上接着跑的组优先，小的先跑：时间不够时能多跑完几组
            item = min(mine, key=lambda it: it["est_seconds"])
        else:
            item = min(pending, key=lambda it: it["est_seconds"])
        item.update(status="running", server=server.name, started_at_hours=round(hours_used(), 4))
        return item


def console_log_path(server, item):
    return (Path(MODEL_INFO[item["model"]]["results_dir"])
            / f"{item['prompt']}_think-{item['think']}_n{item['n']}_{server.name}_console.log")


def run_evaluator(server, item, workers):
    info = MODEL_INFO[item["model"]]
    prefix = f"{server.name} {item['model']} {item['prompt']}/思考{item['think']}"
    command = [sys.executable, str(EVAL_SCRIPT), "--provider", "ollama", "--base-url", server.url,
               "--data-file", str(DATA), "--workers", str(workers), "--timeout", str(REQUEST_TIMEOUT),
               "--out-dir", info["results_dir"], "--model", item["model"], "--prompt", item["prompt"],
               "--think", item["think"], "--n", str(item["n"]), "--show", "1"]
    log_path = console_log_path(server, item)
    started = time.monotonic()
    stopped = {"reason": None}
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n" + datetime.now(timezone.utc).isoformat() + "  " + " ".join(command) + "\n")
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   start_new_session=True)
        RUNNING[server.name] = process

        def on_deadline():
            stopped["reason"] = "deadline"
            stop_process(process, "到达总时长", prefix)

        timer = threading.Timer(max(1.0, seconds_left()), on_deadline)
        timer.daemon = True
        timer.start()
        try:
            for line in process.stdout:
                say(prefix, line)
                log.write(line)
                log.flush()
            code = process.wait()
        finally:
            timer.cancel()
            process.stdout.close()
            RUNNING.pop(server.name, None)
    if STOP.is_set() and stopped["reason"] is None:
        stopped["reason"] = "manual"
    return code, stopped["reason"], time.monotonic() - started


def read_run_summary(item, server):
    file_name, _ = run_file(item["model"], item["prompt"], item["think"], item["n"], server.url)
    path = Path(MODEL_INFO[item["model"]]["results_dir"]) / f"{file_name}_summary.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


def gpu_placement(server, model):
    try:
        loaded = next((m for m in server.api("/api/ps").get("models", [])
                       if model in (m.get("name"), m.get("model"))), None)
        return None if loaded is None else {"size": loaded.get("size"), "size_vram": loaded.get("size_vram")}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}


def run_item(server, item):
    model, prefix = item["model"], f"{server.name} {item['model']} {item['prompt']}/思考{item['think']}"
    caps = MODEL_INFO[model].get("capabilities") or []
    if item["think"] == "on" and caps and "thinking" not in caps:
        item["status"] = "skipped_no_thinking"
        return
    for attempt in (1, 2, 3):
        if STOP.is_set() or seconds_left() < MIN_START_MINUTES * 60:
            item.update(status="stopped_deadline" if attempt > 1 else "skipped_budget",
                        server=item["server"] if attempt > 1 else None)
            return
        try:
            workers = server.ensure(model, item["think"])
        except Exception as e:
            item.update(status="failed", error=f"{type(e).__name__}: {e}")
            note(f"[{prefix}] 准备服务时出错，这组跳过：{type(e).__name__}: {e}")
            return
        if STOP.is_set() or seconds_left() < MIN_START_MINUTES * 60:
            item.update(status="stopped_deadline" if attempt > 1 else "skipped_budget",
                        server=item["server"] if attempt > 1 else None)
            return
        say(prefix, f"开始（第 {attempt} 次）：{workers} 路，已用 {hours_used():.2f} 小时，"
                    f"离停止还有 {seconds_left() / 3600:.2f} 小时")
        code, reason, seconds = run_evaluator(server, item, workers)
        summary = read_run_summary(item, server) if code == 0 else None
        item["minutes"] = round(item["minutes"] + seconds / 60, 1)
        item["workers"] = workers
        item["attempts"].append({"workers": workers, "return_code": code, "stopped": reason,
                                 "minutes": round(seconds / 60, 1), "placement": gpu_placement(server, model),
                                 "errors": summary.get("errors") if summary else None,
                                 "n_done": summary.get("n") if summary else None})
        save_plan_status()
        if reason is not None:
            item["status"] = "stopped_deadline" if reason == "deadline" else "stopped_manual"
            return
        if code == 0 and summary and not summary.get("errors"):
            item["status"] = "completed"
            return
        if code == 0 and summary:  # 有请求失败的条目：再跑一次只补这些；补跑还失败、又是并行时，改成单路
            item["status"] = "completed_with_errors"
            if attempt == 3:
                return
            if attempt == 2 and workers > 1:
                server.single_only.add(model)
            note(f"[{prefix}] {summary.get('errors')} 条请求失败，"
                 f"{'改成单路' if model in server.single_only and workers > 1 else ''}再补跑一次。")
            continue
        if workers > 1 and attempt < 3:
            server.single_only.add(model)
            note(f"[{prefix}] 并行运行失败（退出码 {code}），改成单路重跑。终端输出的最后部分：\n"
                 + tail(console_log_path(server, item), 800))
            continue
        item.update(status="failed", error=f"退出码 {code}")
        note(f"[{prefix}] 运行失败，退出码 {code}，跳过这组。")
        return


def worker(server):
    try:
        while True:
            item = take_item(server)
            if item is None:
                return
            try:
                run_item(server, item)
            except Exception as e:
                import traceback
                item.update(status="failed", error=f"{type(e).__name__}: {e}")
                note(f"[{server.name}] {item['model']} {item['prompt']}/思考{item['think']} 出错：{type(e).__name__}: {e}")
                say(server.name, traceback.format_exc())
            save_plan_status()
            say(server.name, f"→ {item['model']} {item['prompt']}/思考{item['think']}：{item['status']}，"
                             f"本次用了 {item['minutes']} 分钟")
    finally:
        say(server.name, f"这张卡的工作结束，已用 {hours_used():.2f} 小时。")


def print_progress():
    with PLAN_LOCK:
        counts = {}
        for it in ITEMS:
            counts[it["status"]] = counts.get(it["status"], 0) + 1
    say("进度", f"已用 {hours_used():.2f} 小时：" + "，".join(f"{k} {v}" for k, v in sorted(counts.items())))


if OLLAMA_READY and ITEMS:
    threads = [threading.Thread(target=worker, args=(s,), name=s.name, daemon=True) for s in SERVERS]
    for t in threads:
        t.start()
    last_progress = time.monotonic()
    try:
        while any(t.is_alive() for t in threads):
            for t in threads:
                t.join(timeout=5)
            if time.monotonic() - last_progress > 1800:
                print_progress()
                last_progress = time.monotonic()
    except KeyboardInterrupt:
        STOP.set()
        for name, process in list(RUNNING.items()):
            stop_process(process, "手动中断", name)
        raise
    for it in ITEMS:
        if it["status"] in ("pending", "running"):
            it["status"] = "skipped_budget" if it["status"] == "pending" else "stopped_deadline"
    save_plan_status()
    for s in SERVERS:
        try:
            save_json(META_DIR / f"ollama_ps_end_{s.name}.json", s.api("/api/ps"))
        except Exception as e:
            print(f"记录 {s.name} 的 ollama ps 失败：", e)
    save_json(META_DIR / "ollama_servers_history.json",
              {s.name: {"starts": s.history, "single_only": sorted(s.single_only),
                        "parallel_context": s.parallel_context, "load_checks": s.load_log} for s in SERVERS})
    try:
        (META_DIR / "nvidia_smi_after_runs.txt").write_text(
            subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout, encoding="utf-8")
    except Exception as e:
        print("记录 nvidia-smi 失败：", e)
    print_progress()
    note(f"实验结束，已用 {hours_used():.2f} 小时。")
else:
    note("Ollama 没有就绪或没有可跑的组，跳过实验。")

In [ ]:
def pct(x):
    return "—" if x is None else f"{x:.2%}"


def rescore_all():
    for slug_dir in sorted(p for p in OLLAMA_RESULTS.iterdir() if p.is_dir()):
        files = sorted(slug_dir.glob("ollama_*.jsonl"))
        if not files:
            continue
        result = subprocess.run([sys.executable, str(RESCORE_SCRIPT), str(RESCORED / slug_dir.name), *map(str, files)],
                                capture_output=True, text=True)
        (META_DIR / f"rescore_{slug_dir.name}.log").write_text(result.stdout + result.stderr, encoding="utf-8")
        print(result.stdout[-3000:])
        if result.returncode != 0:
            note(f"重新判分 {slug_dir.name} 出错：{result.stderr[-800:]}")


def load_overview():
    rows = []
    for path in sorted(RESCORED.rglob("*_summary.json")):
        s = json.loads(path.read_text(encoding="utf-8"))
        n_target = s.get("target_n") or 0
        first = None
        csv_path = path.with_name(path.name.removesuffix("_summary.json") + ".csv")
        if csv_path.exists():
            with csv_path.open(encoding="utf-8-sig", newline="") as handle:
                marks = [row["correct"] == "1" for row in csv.DictReader(handle) if int(row["idx"]) < N_SUBSET]
            first = (sum(marks) / len(marks), len(marks)) if marks else None
        stem = path.name.removesuffix("_summary.json")
        rows.append({"slug": path.parent.name, "run": stem, "model": s.get("model"), "prompt": s.get("prompt"),
                     "think": s.get("think"), "target": n_target, "done": s.get("n"), "accuracy": s.get("accuracy"),
                     "first_n_accuracy": first[0] if first else None, "first_n_rows": first[1] if first else 0,
                     "invalid": s.get("invalid"), "truncated": s.get("truncated"), "errors": s.get("errors"),
                     "avg_seconds": s.get("avg_seconds"),
                     "avg_output_tokens": round(s["completion_tokens"] / s["n"], 1) if s.get("n") else None,
                     "current_version": any(path.parent.name == info["slug"] for info in MODEL_INFO.values())})
    return rows


def build_summary():
    rescore_all()
    overview = load_overview()
    with open(ROOT / "overview.csv", "w", encoding="utf-8-sig", newline="") as handle:
        if overview:
            writer = csv.DictWriter(handle, fieldnames=list(overview[0]))
            writer.writeheader()
            writer.writerows(overview)

    by_run = {(r["slug"], r["run"]): r for r in overview}
    status_lines = ["| 模型 | 模板 | 思考 | 状态 | 在哪张卡 | 并行 | 跑之前 → 跑之后 | 用时（分钟） |",
                    "|---|---|---|---|---|---:|---|---:|"]
    for it in ITEMS:
        after = "—"
        info = MODEL_INFO.get(it["model"])
        server = next((s for s in SERVERS if s.name == (it["server"] or it["pinned"])), None)
        if info and server:
            name, _ = run_file(it["model"], it["prompt"], it["think"], it["n"], server.url)
            hit = by_run.get((info["slug"], name))
            after = f"{hit['done']}" if hit else "—"
        status_lines.append(f"| {it['model']} | {it['prompt']} | {it['think']} | {it['status']} | "
                            f"{it['server'] or '—'} | {it['workers'] or '—'} | {it['done_before']} → {after} / {it['n']} | "
                            f"{it['minutes']} |")

    combos = [(p, t) for p in ("basic", "cot", "role_cot") for t in ("off", "on")]
    pivot = ["| 模型 | " + " | ".join(f"{p} 思考{t}" for p, t in combos) + " |", "|---|" + "---:|" * len(combos)]
    for model in OLLAMA_MODELS:
        cells = []
        for p, t in combos:
            hits = [r for r in overview if r["model"] == model and r["prompt"] == p and r["think"] == t
                    and r["first_n_rows"]]
            hits.sort(key=lambda r: (r["current_version"], r["first_n_rows"]), reverse=True)
            if not hits:
                cells.append("—")
                continue
            r = hits[0]
            text = pct(r["first_n_accuracy"])
            if r["first_n_rows"] < N_SUBSET:
                text += f"（{r['first_n_rows']} 条）"
            if not r["current_version"]:
                text += "†"
            cells.append(text)
        pivot.append(f"| {model} | " + " | ".join(cells) + " |")

    run_lines = ["| 模型 | 模板 | 思考 | 完成/目标 | 准确率 | 前 200 条 | 无效 | 截断 | 请求失败 | 秒/条 | 输出 token/条 | 结果文件 |",
                 "|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---|"]
    for r in sorted(overview, key=lambda r: (r["model"] or "", r["prompt"] or "", r["think"] or "", r["run"])):
        run_lines.append(f"| {r['model']} | {r['prompt']} | {r['think']} | {r['done']}/{r['target']} | {pct(r['accuracy'])} | "
                         f"{pct(r['first_n_accuracy'])} | {r['invalid']} | {r['truncated']} | {r['errors']} | "
                         f"{r['avg_seconds']} | {r['avg_output_tokens']} | {r['slug']}/{r['run']} |")

    summary_md = "\n\n".join([
        "# SST-2 剩余实验汇总",
        f"总用时 {hours_used():.2f} 小时。开思考的组每张卡 {PARALLEL} 路并行（出过问题的模型改为单路，见“这次 16 组”的并行列）。",
        "## 这次 16 组", "\n".join(status_lines),
        f"## 24 组的前 {N_SUBSET} 条准确率（已用修正后的解析规则重新判分）",
        "\n".join(pivot) + ("\n\n† 来自上次的模型版本（这次下载到的版本不同）。" if any("†" in line for line in pivot) else ""),
        "## 每组明细", "\n".join(run_lines),
        "## 运行记录", "\n".join(f"- [{n['hours']:.2f} 小时] {n['message']}" for n in STATUS["notes"])])
    (ROOT / "summary.md").write_text(summary_md + "\n", encoding="utf-8")
    print(summary_md)


try:
    build_summary()
except Exception as e:
    import traceback
    note(f"汇总时出错（不影响打包）：{type(e).__name__}: {e}")
    print(traceback.format_exc())

In [ ]:
try:
    (META_DIR / "nvidia_smi_end.txt").write_text(
        subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout, encoding="utf-8")
except Exception as e:
    print("记录 nvidia-smi 失败：", e)
note(f"开始打包，已用 {hours_used():.2f} 小时。")

manifest_path = META_DIR / "export_manifest.json"
allowed = {".json", ".jsonl", ".csv", ".tsv", ".md", ".txt", ".log", ".py"}
files = sorted(p for p in ROOT.rglob("*")
               if p.is_file() and not p.is_symlink() and p.suffix.lower() in allowed and p != manifest_path
               and not any(part.startswith(".") for part in p.relative_to(ROOT).parts))
save_json(manifest_path, [{"path": p.relative_to(ROOT).as_posix(), "bytes": p.stat().st_size,
                           "sha256": hashlib.sha256(p.read_bytes()).hexdigest()} for p in files])
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files + [manifest_path]:
        archive.write(path, arcname="sst2_finish/" + path.relative_to(ROOT).as_posix())
print(f"结果包：{ZIP_PATH}，{len(files) + 1} 个文件，{ZIP_PATH.stat().st_size / 2 ** 20:.1f} MB")
unfinished = [it for it in ITEMS if it["status"] != "completed"]
if unfinished:
    print(f"有 {len(unfinished)} 组没有跑完（见 summary.md）。把 zip 发给 Claude，或者把这次的 Output 加为 Input 再运行一次，会接着跑。")
print("在这个版本的 Output 里下载 sst2_finish_results.zip，发给 Claude。")